<b><font size="6" color="#E8800A">Week 11b: Boosting for regression</font></b><br>
<b><font size="4">Training-only selection, one sealed test partition</font></b><br>

Boosting fits learners in sequence. The validation curves show how the base learner, stage count and learning rate change the result.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>AdaBoost</font>](#ada)
* [<font color='#E8800A'>AdaBoost settings</font>](#ada_settings)
* [<font color='#E8800A'>Gradient boosting</font>](#gb)
* [<font color='#E8800A'>Gradient boosting settings</font>](#gb_settings)
* [<font color='#E8800A'>Histogram gradient boosting</font>](#hist)
* [<font color='#E8800A'>Histogram boosting settings</font>](#hist_settings)
* [<font color='#E8800A'>Selection of the boosting model</font>](#sel1)
* [<font color='#E8800A'>The boosting winner</font>](#boost_winner)
* [<font color='#E8800A'>The boosting result</font>](#board)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>Exit ticket</font>](#exit)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

The test rows are sealed, the training folds are prepared once and the earlier learners are written out once, so every score in the notebook is read on the same rows.

In [ ]:
# The notebook folder holds preprocessing.py and course_helpers.py, so it joins the import path first.
import sys
from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Libraries for data handling, figures and parallel work.
import os
import json
import time
import warnings
import joblib
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# scikit-learn: the models, the validation tools and the metrics used below.
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor, BaggingRegressor
from sklearn.linear_model import Lasso, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import ParameterGrid, RandomizedSearchCV, cross_validate, train_test_split, KFold
from sklearn.preprocessing import FunctionTransformer
from sklearn.utils.validation import has_fit_parameter
# Progress bars, then the course helpers: the figure palette, the model log and the scoreboard.
from tqdm.auto import tqdm
from course_helpers import PLOT_BLUE, PLOT_ORANGE, Scoreboard, record, results_before, result_context
# The fold-owned preparation, stacking and blending classes written for this course.
from preprocessing import LockedMemory
from matplotlib_inline.backend_inline import set_matplotlib_formats
from matplotlib.patches import FancyBboxPatch

# One seed for every random choice, then the figure style: white grid, 150 dpi, vector output.
random_state = 33
np.random.seed(random_state)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150
set_matplotlib_formats("svg")
# The exact-message filter also reaches the parallel workers through the environment; the library message is expected, because a validation fold can hold a category the training rows lack.
os.environ["PYTHONWARNINGS"] = "ignore:Found unknown categories:UserWarning"
warnings.filterwarnings("ignore", message="Found unknown categories.*during transform", category=UserWarning)
print("scikit-learn", sklearn.__version__)
from preprocessing import save_boosting_selection

from preprocessing import (
    CachedSearchCV,
    FoldPreprocessor,
    PreparedEstimator,
    load_recipe,
    outer_folds,
    outer_splitter,
    prepared_folds,
)
regression_recipe_log = "../../logs/week_04_feature_work_regression_log.json"


__Step 1:__ Load the frame, seal the test partition, and prepare the training folds.

In [ ]:
# The cars4you table with the feature recipe of the feature-work notebook.
recipe_log = "../../logs/week_04_feature_work_regression_log.json"
frame, recipe = load_recipe("../../data/interim/cars4you.csv", recipe_log)

# Features and target, then the test rows are sealed once: 20 percent of the cars, and no choice below reads them.
features = frame.loc[:, recipe["categorical"] + recipe["numeric"]].copy()
target = frame["price"]
X_train, X_test, y_train, y_test = train_test_split(features, target,
    test_size=0.2, random_state=random_state)
# `prepared` and `raw_prepared` hold, for each of the ten folds, the rows to fit (`X_fit`, `y_fit`) and the rows to score (`X_score`, `y_score`): prepared with the recipe and left raw.
preparation = FoldPreprocessor(recipe)
folds = outer_splitter("regression", random_state=random_state)
prepared = prepared_folds(preparation, list(folds.split(X_train, y_train)), X_train, y_train)
raw_prepared = prepared_folds(FunctionTransformer(), list(folds.split(X_train, y_train)), X_train, y_train)
# The recipe is fitted once on all the training rows. Every model fitted on all the rows learns from `Z_train` and is scored on `Z_test`, the same prepared rows, so no model refits the recipe.
full_prep = clone(preparation).fit(X_train, y_train)
Z_train = full_prep.transform(X_train)
Z_test = full_prep.transform(X_test)
# `model_log` collects every logged result in order; `results` is the folder of saved results and of this notebook's fitted winner.
model_log = []
results_dir = "../../results"
print("training folds:", folds.get_n_splits())

The test partition stays fixed. Validation alone chooses candidates; test columns are descriptive checkpoints. The prepared matrices above belong to their training folds. Stacking, blending and probability calibration need additional internal splits, so their members receive raw rows and fit preparation after those splits.

__Step 2:__ Write the earlier learners once: the stored winner of each earlier algorithm and the best evaluated bagging and forest.

In [ ]:
taught_learners = {
    "Lasso, log price": TransformedTargetRegressor(regressor=Lasso(alpha=0.0001, max_iter=10000), func=np.log, inverse_func=np.exp),
    "Bayesian ridge": BayesianRidge(lambda_1=0.0001, alpha_1=100000, max_iter=2),
    "KNN": KNeighborsRegressor(n_neighbors=5, p=1, weights="distance"),
    "pruned tree, log price": TransformedTargetRegressor(regressor=DecisionTreeRegressor(ccp_alpha=0.0001, criterion="absolute_error", max_depth=None, max_features=None, min_samples_leaf=5, min_samples_split=2, splitter="best", random_state=random_state), func=np.log, inverse_func=np.exp),
    "bagged trees": BaggingRegressor(estimator=DecisionTreeRegressor(random_state=33), n_estimators=105, n_jobs=4, random_state=33),
    "random forest": RandomForestRegressor(min_samples_leaf=2, n_jobs=4, random_state=33),
}

__Step 3:__ Print the stored labels beside the constructors and identify AdaBoost eligibility.

<div class="alert alert-block alert-info">
<code>LockedMemory(location=None, backend="local", mmap_mode=None, compress=False, verbose=1, backend_options=None)</code>

A course class in `preprocessing.py`, beside this notebook.

**Definition:**
A joblib `Memory` with a lock file keyed by the function and arguments. The workers wait for the first computation, then read its cache entry.

**Main Parameters:**
- `location`: the folder of the cache; `None` switches the cache off.
- The other parameters are those of joblib `Memory`.

**Main Methods:**
- `cache(func)`: the cached, locked version of `func`.

**Differences from scikit-learn:**
scikit-learn uses joblib [`Memory`](https://joblib.readthedocs.io/en/latest/user_guide/memory.html), for example in `Pipeline(memory=...)`, which allows concurrent computation of a missing entry. `LockedMemory` prevents repeated preprocessing fits across workers. Without the `filelock` package, locking is skipped: duplicate computation remains possible, with the same result.
</div>

In [ ]:
# Print each stored winner beside the constructor written above, so the two can be compared.
for (name, learner), week in zip(list(taught_learners.items())[:4], range(6, 10)):
    stored = json.loads(Path(results_dir, f"week_{week:02d}.json").read_text(encoding="utf-8"))["regression"]["model"]
    print(f"{name} | week_{week:02d}: {stored}")
    print("constructor:", learner)
# The bagged trees and the forest have no stored label: they are the best tested configurations of the previous notebook.
for name, learner in list(taught_learners.items())[4:]:
    print(f"{name} | best evaluated in the previous week")
    print("constructor:", learner)
# AdaBoost refits its base on reweighted rows, so every learner is checked for a `sample_weight` argument in `fit`.
sample_weight_support = {name: has_fit_parameter(learner, "sample_weight") for name, learner in taught_learners.items()}
# AdaBoost.R2 draws weighted rows itself, so every learner is eligible; ensembles shrink to 16 members because every stage refits the base.
eligible_ada = {name: clone(learner).set_params(**{key: value for key, value in dict(n_estimators=16).items() if key in learner.get_params(deep=False)}) for name, learner in taught_learners.items()}
# `ada_members` holds the eligible learners; each is fitted on rows the recipe has already prepared.
ada_members = dict(eligible_ada)
# Say which learners can serve as a base and which cannot.
print("base fit sample_weight signatures:", sample_weight_support)
print("AdaBoost eligible:", list(eligible_ada))
print("AdaBoost excluded:", [name for name in taught_learners if name not in eligible_ada])
# `member_memory` stores the predictions of each combiner member for identical fit and score rows, so a combination that reuses a member never refits it; it is a `LockedMemory`, so workers that need the same entry wait for one computation.
member_memory = LockedMemory(".ensemble_member_cache", verbose=0)

AdaBoost.R2 draws rows from its own sample-weight distribution before fitting each regressor. Unlike the classifier, `AdaBoostRegressor` needs no base `fit` argument named `sample_weight`, so all taught regressors qualify, including KNN and the log-target wrappers. Bagged trees and the forest use 16 members instead of 105 and 100 because each stage refits the base learner. The recipe prepares rows once per fold and once on all training rows; stages draw weighted samples from those prepared rows. [AdaBoostRegressor API](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostRegressor.html), [weighted bootstrap implementation](https://github.com/scikit-learn/scikit-learn/blob/1.9.0/sklearn/ensemble/_weight_boosting.py)

# <font color='#E8800A'>AdaBoost</font> <a class="anchor" id="ada"></a>
[Back to TOC](#toc)

The next learner receives a changed view of the errors. AdaBoost classification increases the relative weight of misclassified rows; AdaBoost.R2 uses a normalised prediction loss and a weighted median of member predictions.

__Step 4:__ Fit a stump to six toy rows and compute the first weight update by hand.

In [ ]:
# Six rows with one input and a numeric response, all starting with the same weight.
toy_x = np.arange(6).reshape(-1, 1)
toy_y = np.array([1., 2., 1., 6., 9., 7.])
toy_weights = np.full(6, 1 / 6)
# One split is fitted with the row weights; each row's absolute error is the distance to its prediction.
toy_stump = DecisionTreeRegressor(max_depth=1, random_state=random_state).fit(toy_x, toy_y, sample_weight=toy_weights)
toy_error = np.abs(toy_y - toy_stump.predict(toy_x))
# Dividing by the largest error puts every row's loss between zero and one.
toy_loss = toy_error / toy_error.max()
toy_average_loss = np.dot(toy_weights, toy_loss)
# The weighted mean loss sets beta, which shrinks the weights of well-predicted rows in the update.
toy_beta = toy_average_loss / (1 - toy_average_loss)
# Rows with a higher loss keep more relative weight after normalising; the table sets old and next weights side by side.
toy_next = toy_weights * toy_beta ** (1 - toy_loss)
toy_next /= toy_next.sum()
print(pd.DataFrame({"response": toy_y, "prediction": toy_stump.predict(toy_x), "normalised loss": toy_loss, "old weight": toy_weights, "next weight": toy_next}).round(4).to_string(index=False))
print("weighted loss:", toy_average_loss, "beta:", toy_beta)

For a linear AdaBoost.R2 loss, $L_i=|y_i-h(x_i)|/\max_j|y_j-h(x_j)|$. Here $i$ and $j$ index toy rows, $y_i$ is the observed response, $x_i$ is its input and $h$ is the fitted stump. The weighted mean loss is $E=\sum_i w_iL_i$, with row weight $w_i$. Set $\beta=E/(1-E)$ and update each weight by $w_i\beta^{1-L_i}$ before normalising; the higher-loss rows retain more relative weight. This is one update, not the entire ensemble prediction.

__Step 5:__ Draw the fitted toy stump, including every node.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
plot_tree(toy_stump, feature_names=["toy position"], filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title("AdaBoost: the first toy learner")
plt.tight_layout()
plt.show()

**The stump predicts 1.3333 for the left rows and 7.3333 for the right rows, so the weighted loss is 0.4667 and beta is 0.875.** Rows with normalised loss 0.2 fall from 0.1667 to 0.1607 while the row with loss 1.0 rises to 0.1788, so the next learner is fitted harder to the rows that this one predicted worst.

__Step 6:__ Draw three stages of AdaBoost on the six toy rows: marker area is the row weight, orange marks a row that the stump of the stage gets wrong, and each title gives the stage weight, the size of that stump's vote in the final answer.

In [ ]:
# Start from equal row weights; `toy_x` and `toy_y` are the six toy rows of the step above.
stage_weights = np.full(6, 1 / 6)
# `grid` is a fine set of positions to draw each stump as a step function.
grid = np.linspace(0, 5, 100).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
# One panel per stage: fit a stump with the current weights, then draw rows sized by weight (orange for the worst-predicted row) and the stump itself.
for stage, axis in tqdm(enumerate(axes, start=1), desc='stage, axis', leave=True, position=0):
    stump = DecisionTreeRegressor(max_depth=1, random_state=random_state).fit(toy_x, toy_y, sample_weight=stage_weights)
    loss = np.abs(toy_y - stump.predict(toy_x))
    loss = loss / loss.max()
    average_loss = np.dot(stage_weights, loss)
    beta = average_loss / (1 - average_loss)
    axis.scatter(toy_x[:, 0], toy_y, s=3000 * stage_weights, c=np.where(loss == 1, PLOT_ORANGE, PLOT_BLUE), edgecolor="white", zorder=3)
    axis.plot(grid[:, 0], stump.predict(grid), color="black", linestyle="--", drawstyle="steps-mid")
    axis.set(title=f"stage {stage}: stage weight {np.log(1 / beta):.2f}", xlabel="toy position")
    print(f"stage {stage}: weighted loss {average_loss:.4f}, stage weight {np.log(1 / beta):.4f}, row weights {stage_weights.round(3)}")
    # Rows with a higher loss keep more weight for the next stump.
    stage_weights = stage_weights * beta ** (1 - loss)
    stage_weights /= stage_weights.sum()
axes[0].set_ylabel("response")
plt.tight_layout()
plt.show()

**The stage weight, the size of a stump's vote, falls from 0.1335 at stage 1 to 0.0385 at stage 2 and 0.0114 at stage 3, because the weighted loss rises from 0.4667 to 0.4904 and 0.4972.** Marker area shows each row's weight; the row the stump fits worst is orange, and after each stage the rows with a higher loss keep more weight.

__Step 7:__ Draw one AdaBoost stage as a flow: what goes in, what is fitted and what is passed on.

In [ ]:
# Each box is a patch on a blank axis: what goes into one AdaBoost stage, what is fitted and what is passed on. Arrows show the order inside one stage.
stage_boxes = ['row weights w\n(they sum to one)', 'fit a learner h\non the weighted rows', 'weighted error E\nand stage weight alpha', 'raise the weights of\nthe wrong rows: new w']
fig, ax = plt.subplots(figsize=(12, 2.8))
for position, label in enumerate(stage_boxes):
    ax.add_patch(FancyBboxPatch((position * 3.1, 0), 2.5, 1.3, boxstyle="round,pad=0.06", facecolor="white", edgecolor=PLOT_BLUE, linewidth=2))
    ax.text(position * 3.1 + 2.5 / 2, 0.65, label, ha="center", va="center", fontsize=10)
    if position:
        ax.annotate("", xy=(position * 3.1, 0.65), xytext=(position * 3.1 - 0.6, 0.65), arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5))
# The last box feeds the first box of the next stage.
ax.plot([10.55, 10.55, 1.25, 1.25], [-0.08, -0.45, -0.45, -0.2], color="black", linewidth=1.5)
ax.annotate("", xy=(1.25, -0.08), xytext=(1.25, -0.25), arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5))
ax.text((9.3 + 2.5) / 2, -0.75, "the new weights are the input of the next stage", ha="center", fontsize=10)
ax.set(xlim=(-0.3, 9.3 + 2.5 + 0.3), ylim=(-1.1, 1.6))
ax.axis("off")
plt.tight_layout()
plt.show()

**One AdaBoost stage takes the row weights in and passes updated weights and a stage weight on.** Follow the arrows: the learner is fitted to the weighted rows, its weighted error sets its stage weight, and the wrong rows gain weight before the next stage.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostRegressor.html'>sklearn.ensemble.AdaBoostRegressor(estimator=None, n_estimators=50, learning_rate=1.0, loss='linear', random_state=None)</a>

**Definition:**
Fits a sequence of base regressors, each on rows drawn in proportion to weights that grow with the earlier errors (AdaBoost.R2), and combines them by a weighted median.

**Main Parameters:**
- `estimator`: base regressor.
- `n_estimators`: maximum number of stages.
- `learning_rate`: multiplier on the stage weights.
- `loss`: shape that maps relative error to the weight update.
- `random_state`: seed of the row draws.

**Main Attributes (after fitting):**
- `estimators_`: the fitted stages.
- `estimator_weights_`: the weight of each stage.
- `estimator_errors_`: the average loss of each stage.

**Main Methods:**
- `fit(X, y, sample_weight=None)`: fit the stages in sequence.
- `predict(X)`: the weighted median.
- `staged_predict(X)`: the prediction after each stage.
</div>

The reference uses Bayesian ridge as its base learner; the sweeps below change one setting of that reference, and the estimator sweep compares every eligible base learner.

__Step 8:__ Score the AdaBoostRegressor reference on the ten prepared training folds.

In [ ]:
ada_reference = AdaBoostRegressor(random_state=random_state)
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(ada_reference).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
ada_default_train = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
ada_default_validation = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]

__Step 9:__ Fit the AdaBoostRegressor reference on all the training rows.

In [ ]:
ada_default = clone(ada_reference).fit(Z_train, y_train)
ada_default_configuration = f"AdaBoostRegressor(" + ", ".join([f"{setting_name}={setting_value}" for setting_name, setting_value in ada_reference.get_params(deep=False).items() if setting_name not in ("estimator", "random_state") and setting_value != AdaBoostRegressor().get_params(deep=False)[setting_name]]) + ")"
ada_complete_rows = []

__Step 10:__ Log the AdaBoostRegressor reference.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": ada_default_configuration, "train": ada_default_train, "validation": ada_default_validation,
    "model": ada_default},
    Z_test, y_test, "untuned", metric=mean_absolute_error, label="MAE")

# <font color='#E8800A'>AdaBoost settings</font> <a class="anchor" id="ada_settings"></a>
[Back to TOC](#toc)

Each sweep changes one AdaBoost setting from the reference and scores every value on the same ten prepared training folds. The section closes on the best configuration actually evaluated; interactions need the later joint search.

| Parameter | Meaning |
| --- | --- |
| `estimator` | Base learner refitted at every stage. |
| `n_estimators` | Maximum number of stages. |
| `learning_rate` | Multiplier on each stage's weight and on its reweighting of rows. |
| `loss` | Shape that maps relative error to the update (AdaBoost.R2 only). |

[Parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostRegressor.html)

__Step 11:__ Change only AdaBoost `estimator` on the fixed training folds, using the eligible learners of `ada_members`.

### `AdaBoost: estimator`

AdaBoost.R2 samples training rows according to their weights before fitting each learner. The default `None` builds a depth-three regression tree; no `sample_weight` argument is required. A stronger base is expected to leave less error to correct while costing more per stage. The candidates are the eligible taught base estimators with their own winning settings fixed, and KNN and both target-transformation wrappers remain eligible.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
ada_estimator_values = list(ada_members.values())
# One row per value will hold its scores, its seconds and its fitted model.
ada_estimator_rows = []
for name, value in tqdm(list(zip(ada_members, ada_estimator_values)), desc="AdaBoostRegressor.estimator", unit="value"):
    started = time.perf_counter()
    candidate = clone(ada_reference).set_params(estimator=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    ada_estimator_rows.append({"configuration": f"AdaBoostRegressor(" + ", ".join([f"estimator={name}"] + [f"{setting_name}={setting_value}" for setting_name, setting_value in candidate.get_params(deep=False).items() if setting_name not in ("estimator", "random_state") and setting_value != AdaBoostRegressor().get_params(deep=False)[setting_name]]) + ")", "setting": name, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
ada_estimator_sweep = pd.DataFrame(ada_estimator_rows)
ada_complete_rows.extend(ada_estimator_rows)
ada_estimator_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
ada_estimator_winner = ada_estimator_sweep.loc[ada_estimator_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": ada_estimator_winner["configuration"], "train": ada_estimator_winner["train"],
    "validation": ada_estimator_winner["validation"], "model": ada_estimator_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**Changing only the base learner gives the lowest validation MAE to bagged trees: 2,307.40 EUR against 4,797.14 EUR for the default depth-three tree.** Later isolated sweeps return to that default tree.

AdaBoost reweights earlier errors, so a stage contributes where its base learner still errs on the weighted training rows. For standard binary AdaBoost with learning rate one, let $\varepsilon_t$ be the weighted training error at stage $t$ and $\gamma_t=\tfrac{1}{2}-\varepsilon_t$ its edge over guessing. After $T$ stages the combined classifier's training error is at most $\prod_{t=1}^{T} 2\sqrt{\varepsilon_t(1-\varepsilon_t)}\le\exp\left(-2\sum_{t=1}^{T}\gamma_t^2\right)$ ([Freund and Schapire, 1997](https://doi.org/10.1006/jcss.1997.1504)). Here $t$ indexes stages, $T$ counts them, the product contains one factor per stage, the sum adds their squared edges and $\exp$ is the exponential function. A positive edge lowers the bound; a larger edge lowers it more.

A weak learner is somewhat better than chance and successive stages can help if they retain a positive edge after reweighting. An elaborate learner may fit almost perfectly at the first stage, receiving a very large weight and leaving little to correct; scikit-learn stops when $\varepsilon_t=0$. Or it may repeat nearly the same function after reweighting, adding cost without new edge.

Unstable does not mean weak: an unpruned deep tree fits its training rows perfectly, leaving nothing to reweight. Bagging averages away its variation; boosting does not.

For regression, AdaBoost.R2 ([Drucker, 1997](https://mlanthology.org/icml/1997/drucker1997icml-improving/)) assigns each row a loss between 0 and 1 relative to the stage's largest error. Its row-weighted average must stay below 0.5; zero average loss ends boosting.

Sources: [Freund and Schapire, 1997](https://doi.org/10.1006/jcss.1997.1504) and the [scikit-learn AdaBoost guide](https://scikit-learn.org/stable/modules/ensemble.html#adaboost).

__Step 12:__ Fit each base learner alone and boosted with the AdaBoost defaults on the ten prepared folds, and score every stage.

In [ ]:
# From the weakest learner to the most elaborate: the shallow trees, the unpruned tree and the learners of the estimator sweep.
comparison_learners = {
    "stump": DecisionTreeRegressor(max_depth=1, random_state=random_state),
    "default tree": DecisionTreeRegressor(max_depth=3, random_state=random_state),
    "deep tree": DecisionTreeRegressor(random_state=random_state),
    **ada_members}
# The AdaBoost defaults, 50 stages at a learning rate of 1.0; only the base learner changes.
default_boost = AdaBoostRegressor(n_estimators=50, learning_rate=1.0, random_state=random_state)
comparison_rows, staged_curves = [], {}
for name, learner in tqdm(comparison_learners.items(), desc="base learners", unit="learner", leave=True, position=0):
    folds_used = prepared
    # Alone: the ten fold fits run at the same time and are scored one by one afterwards.
    started = time.perf_counter()
    alone_fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(learner).fit)(fold["X_fit"], fold["y_fit"]) for fold in folds_used)
    alone_scores = np.array([mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(folds_used, alone_fits)])
    alone_seconds = time.perf_counter() - started
    # Boosted: the same folds, with the learner as the base of AdaBoost.
    started = time.perf_counter()
    boosted_fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(default_boost).set_params(estimator=learner).fit)(fold["X_fit"], fold["y_fit"]) for fold in folds_used)
    boosted_seconds = time.perf_counter() - started
    # Keep actual fitted stages; a curve stops when the shortest fold stops.
    staged = []
    for fold, fitted in zip(folds_used, boosted_fits):
        answers = list(fitted.staged_predict(fold["X_score"]))
        staged.append([mean_absolute_error(fold["y_score"], answer) for answer in answers])
    boosted_scores = np.array([curve[-1] for curve in staged])
    common_stages = min(map(len, staged))
    staged_curves[name] = np.mean([curve[:common_stages] for curve in staged], axis=0)
    # One row per learner: the scores, the gain, the spread over the folds, the stages kept, the error of the first stage and the seconds.
    comparison_rows.append({"learner": name, "alone MAE": alone_scores.mean(), "boosted MAE": boosted_scores.mean(),
        "gain": alone_scores.mean() - boosted_scores.mean(),
        "SD alone": alone_scores.std(ddof=1), "SD boosted": boosted_scores.std(ddof=1),
        "stages kept": np.mean([len(fitted.estimators_) for fitted in boosted_fits]),
        "first-stage error": np.mean([fitted.estimator_errors_[0] for fitted in boosted_fits]),
        "seconds alone": alone_seconds, "seconds boosted": boosted_seconds})
base_comparison = pd.DataFrame(comparison_rows).set_index("learner")

__Step 13:__ Print the scores, the gain, the spread, the stages kept, the first-stage error and the seconds of every base learner.

In [ ]:
# The scores alone and boosted, the gain, the spread over the ten folds, the stages kept, the weighted error of the first stage and the seconds.
display(base_comparison.round(4))

A positive gain means a lower MAE: it is the boosted score minus the score alone, counted in that direction. Standard deviations across the ten folds measure sensitivity to changing training rows, in MAE units; they are not error scores. Stages kept is mean `len(estimators_)` after stopping; first-stage error is mean `estimator_errors_[0]`, the first stage's weighted error. Seconds measure wall time for the ten fold fits at four workers.

**Boosting lowers the MAE of the deep tree by 697.41 EUR, from 3,053.52 to 2,356.11, and the MAE of the log-price pruned tree by 369.52, while it raises the MAE of the stump by 3,456.34 EUR to 9,192.60 and that of the default tree by 718.95 to 4,797.14.** The two linear models also lose, Lasso by 549.23 EUR and Bayesian ridge by 819.15, and they keep 12.6 and 10.0 stages on average. KNN gains 27.43, bagged trees 67.44 and the forest 46.44. The first-stage error is 0.0170 for the deep tree and 0.0313 for the pruned tree, against 0.0590 for the stump. The fold standard deviation of the deep tree falls from 170.49 to 127.31, while that of the stump rises from 210.60 to 918.22.

__Step 14:__ Plot the score after every stage and the gain of every base learner.

In [ ]:
# Each curve starts at the first fitted stage and ends when the shortest fold stops.
fig, ax = plt.subplots(figsize=(9, 5))
colours = sns.color_palette("colorblind", len(staged_curves))
for colour, (name, curve) in zip(colours, staged_curves.items()):
    ax.plot(np.arange(1, len(curve) + 1), curve, color=colour, linewidth=2, label=name)
    ax.scatter([len(curve)], [curve[-1]], color=colour, marker="x" if len(curve) < 50 else "o", zorder=3)
ax.legend(fontsize=8)
ax.set(xlabel="actual boosting stages", ylabel="Validation MAE", title="AdaBoost: validation after each fitted stage")
plt.tight_layout()
plt.show()

**In the left panel the deep tree falls from 3,053.52 EUR alone to 2,356.11 boosted and the pruned tree from 2,710.20 to 2,340.68, while the stump rises from 5,736.25 to 9,192.60.** In the right panel the bars are the gains of the table above, 697.41 for the deep tree and -3,456.34 for the stump.

__Step 15:__ Change only AdaBoost `n_estimators` on the fixed training folds, with the sweep list `[1, 5, 10, 30, 50, 70, 90, 110, 130, 150]`.

### `AdaBoost: n_estimators`

Each stage fits another base learner, up to this limit; the library default is `50`. More stages can correct remaining errors, but validation may flatten or worsen as the model follows noise. Fitting stops early if the update cannot continue. The candidate list is `[1, 5, 10, 30, 50, 70, 90, 110, 130, 150]`.

In [ ]:
# The stage counts to report; one fit per fold at the largest count gives the prediction after every smaller count as well.
ada_n_estimators_values = [1, 5, 10, 30, 50, 70, 90, 110, 130, 150]
ada_n_estimators_rows = []
ada_stage_count = max(ada_n_estimators_values)
# Arrays of shape folds by stages: the training and validation score of each fold after each stage.
ada_train_by_fold = np.zeros((len(prepared), ada_stage_count))
ada_validation_by_fold = np.zeros((len(prepared), ada_stage_count))
# The ten fold fits run at the same time.
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(ada_reference).set_params(n_estimators=ada_stage_count).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
# `staged_predict` yields the prediction after stage 1, 2, 3 and so on: score each stage on the fitted rows and on the validation rows.
for position, (fold, fitted) in tqdm(list(enumerate(zip(prepared, fits))), desc="AdaBoostRegressor staged scores", unit="fold"):
    on_fit = list(fitted.staged_predict(fold["X_fit"]))
    on_score = list(fitted.staged_predict(fold["X_score"]))
    for stage in range(ada_stage_count):
        used = min(stage, len(on_fit) - 1)
        ada_train_by_fold[position, stage] = mean_absolute_error(fold["y_fit"], on_fit[used])
        ada_validation_by_fold[position, stage] = mean_absolute_error(fold["y_score"], on_score[used])
    # Keep the first fold's staged answers for the check below.
    if position == 0:
        first_staged = on_score
# Check: a fresh fit with fewer stages must predict exactly like the early part of the long fit, at each count before the last.
for count in tqdm(ada_n_estimators_values[:-1], desc='ada n estimators values[:-1]', leave=True, position=0):
    fresh = clone(ada_reference).set_params(n_estimators=count).fit(prepared[0]["X_fit"], prepared[0]["y_fit"])
    assert np.allclose(fresh.predict(prepared[0]["X_score"]), first_staged[min(count, len(first_staged)) - 1]), count
print("staged predictions equal fresh fits at", ada_n_estimators_values[:-1], "stages")
# One row per reported count: its fold scores come from the staged arrays, and its all-rows fit is scored on the sealed test rows.
for value in tqdm(ada_n_estimators_values, desc='AdaBoostRegressor.n_estimators', unit='value', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(ada_reference).set_params(n_estimators=value)
    train_scores = list(ada_train_by_fold[:, value - 1])
    validation_scores = list(ada_validation_by_fold[:, value - 1])
    deployed = clone(candidate).fit(Z_train, y_train)
    ada_n_estimators_rows.append({"configuration": f"AdaBoostRegressor(" + ", ".join([f"{setting_name}={setting_value}" for setting_name, setting_value in candidate.get_params(deep=False).items() if setting_name not in ("estimator", "random_state") and setting_value != AdaBoostRegressor().get_params(deep=False)[setting_name]]) + ")", "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per reported count, with the mean fold scores, the test score and the seconds spent on its all-rows fit.
ada_n_estimators_sweep = pd.DataFrame(ada_n_estimators_rows)
ada_complete_rows.extend(ada_n_estimators_rows)
ada_n_estimators_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
ada_n_estimators_winner = ada_n_estimators_sweep.loc[ada_n_estimators_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": ada_n_estimators_winner["configuration"], "train": ada_n_estimators_winner["train"],
    "validation": ada_n_estimators_winner["validation"], "model": ada_n_estimators_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**The default tree scores validation MAE 4,797.14 EUR at 50 stages; the best tested count is 10, scoring 3,902.42 EUR.** More stages do not help throughout this sweep.

__Step 16:__ Plot training and validation MAE against the number of AdaBoost stages.

In [ ]:
# Lines: the mean score of the ten folds after every stage. Dots: the counts reported in the table above.
fig, ax = plt.subplots(figsize=(7, 5))
ada_stages = np.arange(1, ada_stage_count + 1)
ax.plot(ada_stages, ada_train_by_fold.mean(axis=0), color=PLOT_ORANGE, linewidth=2, label="training (ten folds)")
ax.plot(ada_stages, ada_validation_by_fold.mean(axis=0), color=PLOT_BLUE, linewidth=2, label="validation (ten folds)")
ax.scatter(ada_n_estimators_sweep["setting"], ada_n_estimators_sweep["train_MAE"], color=PLOT_ORANGE, zorder=3)
ax.scatter(ada_n_estimators_sweep["setting"], ada_n_estimators_sweep["cv_MAE"], color=PLOT_BLUE, zorder=3)
ax.set(xlabel="n_estimators", ylabel="MAE", title="AdaBoostRegressor: train against validation")
ax.legend()
plt.tight_layout()
plt.show()

__Step 17:__ Change only AdaBoost `learning_rate` on the fixed training folds, with the sweep list `[0.1, 0.5, 1.0]`.

### `AdaBoost: learning_rate`

AdaBoost.R2 uses this rate in its row-weight update and stage weights; the default is `1.0`. It changes subsequent weighted sampling and the weighted median, rather than shrinking each numeric tree prediction as gradient boosting does. A smaller rate is expected to change weights more slowly and often need more stages. The candidate list is `[0.1, 0.5, 1.0]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
ada_learning_rate_values = [0.1, 0.5, 1.0]
# One row per value will hold its scores, its seconds and its fitted model.
ada_learning_rate_rows = []
for value in tqdm(ada_learning_rate_values, desc="AdaBoostRegressor.learning_rate", unit="value"):
    started = time.perf_counter()
    candidate = clone(ada_reference).set_params(learning_rate=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    ada_learning_rate_rows.append({"configuration": f"AdaBoostRegressor(" + ", ".join([f"{setting_name}={setting_value}" for setting_name, setting_value in candidate.get_params(deep=False).items() if setting_name not in ("estimator", "random_state") and setting_value != AdaBoostRegressor().get_params(deep=False)[setting_name]]) + ")", "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
ada_learning_rate_sweep = pd.DataFrame(ada_learning_rate_rows)
ada_complete_rows.extend(ada_learning_rate_rows)
ada_learning_rate_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
ada_learning_rate_winner = ada_learning_rate_sweep.loc[ada_learning_rate_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": ada_learning_rate_winner["configuration"], "train": ada_learning_rate_winner["train"],
    "validation": ada_learning_rate_winner["validation"], "model": ada_learning_rate_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**With the default tree and 50 stages fixed, validation MAE is 3,875.20 EUR at rate 0.1 and 4,797.14 EUR at rate 1.0.** The smaller rate helps this reference, but the separate interaction curves determine how many stages it needs.

__Step 18:__ Change only AdaBoost `loss` on the fixed training folds, with the sweep list `['linear', 'square', 'exponential']`.

### `AdaBoost: loss`

AdaBoost.R2 scales row errors by the largest error, then applies a linear, squared or exponential transformation. The default is `linear`. Squaring small relative errors reduces their influence, so larger errors are expected to dominate the next row-weight update. The candidate list is `['linear', 'square', 'exponential']`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
ada_loss_values = ['linear', 'square', 'exponential']
# One row per value will hold its scores, its seconds and its fitted model.
ada_loss_rows = []
for value in tqdm(ada_loss_values, desc="AdaBoostRegressor.loss", unit="value"):
    started = time.perf_counter()
    candidate = clone(ada_reference).set_params(loss=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    ada_loss_rows.append({"configuration": f"AdaBoostRegressor(" + ", ".join([f"{setting_name}={setting_value}" for setting_name, setting_value in candidate.get_params(deep=False).items() if setting_name not in ("estimator", "random_state") and setting_value != AdaBoostRegressor().get_params(deep=False)[setting_name]]) + ")", "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
ada_loss_sweep = pd.DataFrame(ada_loss_rows)
ada_complete_rows.extend(ada_loss_rows)
ada_loss_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
ada_loss_winner = ada_loss_sweep.loc[ada_loss_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": ada_loss_winner["configuration"], "train": ada_loss_winner["train"],
    "validation": ada_loss_winner["validation"], "model": ada_loss_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**With the default tree, 50 stages and rate 1.0 fixed, exponential gives the lowest validation MAE of the tested losses: 4,775.95 EUR.** The linear default scores 4,797.14 EUR.

__Step 19:__ Plot training and validation MAE for every other swept AdaBoost setting, one panel each.

In [ ]:
# The sweep tables of this family, each with its list of swept values for the axis labels.
ada_settings_tables = {"learning_rate": (ada_learning_rate_sweep, ada_learning_rate_values), "loss": (ada_loss_sweep, ada_loss_values)}
# One panel per setting: training and validation score at every swept value, side by side.
fig, axes = plt.subplots(1, 2, figsize=(8, 3.6), squeeze=False)
for axis, (setting, (table, values)) in zip(axes.ravel(), ada_settings_tables.items()):
    positions = np.arange(len(table))
    axis.plot(positions, table["train_MAE"], marker="o", color=PLOT_ORANGE, label="training")
    axis.plot(positions, table["cv_MAE"], marker="o", color=PLOT_BLUE, label="validation")
    axis.set(title=setting, xticks=positions, xticklabels=[str(value) for value in values], ylabel="MAE")
    axis.tick_params(axis="x", labelrotation=20)
axes.ravel()[0].legend()
# Hide the panels left over in the last row.
for axis in axes.ravel()[len(ada_settings_tables):]:
    axis.set_visible(False)
plt.tight_layout()
plt.show()

**The rate sweep keeps the default base and 50 stages fixed: validation MAE changes from 3,875.20 EUR at 0.1 to 4,797.14 EUR at 1.0.** Read this as a one-setting comparison, separate from the rate-by-stage study.

__Step 20:__ Select the best tested AdaBoost configuration from the existing sweep rows by validation score.

A smaller learning rate makes each correction smaller, so it often needs more stages. These curves vary the rate and stage budget together while keeping the default base learner fixed. Validation decides whether extra stages help: noise, overfitting and early termination can break the pattern. Equal rate-times-stage products need not produce equal models.

__Step 21:__ Read the validation score after each actual stage at each learning rate.

In [ ]:
# The default base is fixed; this separate interaction varies the rate and stage budget.
ada_curve_rates = ada_learning_rate_values
ada_rate_curves = {}
for rate in tqdm(ada_curve_rates, desc="AdaBoost learning-rate interaction", unit="rate"):
    fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(ada_reference).set_params(learning_rate=rate, n_estimators=150).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
    curves = [[mean_absolute_error(fold["y_score"], answer) for answer in fitted.staged_predict(fold["X_score"])] for fold, fitted in zip(prepared, fits)]
    actual_stages = min(map(len, curves))
    ada_rate_curves[rate] = np.mean([curve[:actual_stages] for curve in curves], axis=0)
fig, ax = plt.subplots(figsize=(8, 5))
for rate, curve in ada_rate_curves.items():
    ax.plot(np.arange(1, len(curve) + 1), curve, linewidth=2, label=f"learning rate {rate}")
    ax.scatter([len(curve)], [curve[-1]], marker="x" if len(curve) < 150 else "o", zorder=3)
ax.set(xlabel="actual boosting stages", ylabel="validation MAE", title="AdaBoost: rate and fitted stages")
ax.legend()
plt.tight_layout()
plt.show()
print("Each curve ends when its shortest fold stops; x marks early termination.")
print(pd.DataFrame([{"learning_rate": rate, "actual stages": len(curve), "best stage": int(np.argmin(curve)) + 1, "best validation MAE": curve.min()} for rate, curve in ada_rate_curves.items()]).round(4).to_string(index=False))

**At rate 0.1, the best plotted validation MAE is 3,833.38 EUR at 78 stages; rate 0.5 reaches 3,830.79 EUR at 11 stages.** The similar errors need different numbers of corrections.

__Step 22:__ Select the best complete AdaBoost configuration actually evaluated.

In [ ]:
ada_complete = pd.DataFrame(ada_complete_rows)
ada_best = ada_complete.loc[ada_complete["cv_MAE"].idxmin()]
ada_best_parameters = ada_best["parameters"]
ada_best_configuration = clone(ada_reference).set_params(**ada_best_parameters)
print("best tested AdaBoost configuration:", ada_best["configuration"])

**The best complete AdaBoost configuration evaluated in the isolated sweeps uses bagged trees, with validation MAE 2,307.40 EUR.** The joint search later tests parameter combinations rather than combining separate sweep winners by assumption.

# <font color='#E8800A'>Gradient boosting</font> <a class="anchor" id="gb"></a>
[Back to TOC](#toc)

Gradient boosting also adds learners in sequence, but it does not reweight rows. Each new regression tree is fitted to the negative gradient of the loss at the predictions of the earlier trees, called the pseudo-residual. For squared error this is the residual, so a small price example makes the correction visible. [Gradient boosting guide](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosting)

Both methods add one learner at a time, but differ in stage inputs and combination rules.

| | AdaBoost.R2 | Gradient boosting |
|---|---|---|
| What a stage fits | A base learner fitted to rows sampled in proportion to weights that grow with earlier errors | A regression tree fitted to pseudo-residuals: the negative loss gradient at earlier predictions |
| Loss | Binary AdaBoost fits the exponential loss by forward stagewise fitting ([Friedman, Hastie and Tibshirani, 2000](https://doi.org/10.1214/aos/1016218223)); AdaBoost.R2 uses relative error shaped by `loss` | Any differentiable loss: `squared_error`, `absolute_error`, `huber` and `quantile` for regression, `log_loss` and `exponential` for classification |
| Base learner | Any regressor: resampling needs no `sample_weight` | A regression tree, whatever the loss |
| How stages combine | Stage weights come from their average losses; prediction is the members' weighted median | A constant plus the trees, each scaled by the same learning rate |
| What a row hands to the next stage | A higher sampling probability after large relative error | A target: its residual under squared error, its residual sign under absolute error |

Gradient boosting retains AdaBoost's stagewise construction while allowing different losses. A regression tree fits the pseudo-residual, the direction of fastest loss decrease. Under `squared_error` this is $r_i=y_i-F(x_i)$, where $i$ indexes rows, $y_i$ is the observed response and $F(x_i)$ the earlier trees' prediction; under `absolute_error` it is the residual's sign. The classification notebook fits the same kind of tree to $y_i-p_i$, the label minus the current predicted probability, under `log_loss`.

__Step 23:__ Draw three stages of gradient boosting on twelve toy rows: the prediction after the earlier trees, the residuals and the tree fitted to them.

In [ ]:
# Twelve rows of one smooth curve; `grid` is a fine set of positions for drawing each tree as a step function.
sketch_x = np.linspace(0, 1, 12).reshape(-1, 1)
sketch_y = 2 + np.sin(5 * sketch_x[:, 0])
grid = np.linspace(0, 1, 200).reshape(-1, 1)
# Stage zero predicts the mean everywhere; each stage then adds a small tree fitted to what is left over.
prediction = np.full(12, sketch_y.mean())
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True)
# Top row: the prediction after the earlier trees (the mean at the start) with an arrow from it to each response (the residual). Bottom row: the tree fitted to those residuals.
for stage in tqdm(range(3), desc='stage', leave=True, position=0):
    residual = sketch_y - prediction
    tree = DecisionTreeRegressor(max_depth=2, random_state=random_state).fit(sketch_x, residual)
    axes[0, stage].scatter(sketch_x[:, 0], sketch_y, color=PLOT_BLUE, zorder=3)
    axes[0, stage].step(sketch_x[:, 0], prediction, where="mid", color=PLOT_ORANGE, linewidth=2)
    axes[0, stage].quiver(sketch_x[:, 0], prediction, np.zeros(12), residual, angles="xy", scale_units="xy", scale=1, width=0.004, color="gray")
    axes[0, stage].set_title("starting prediction: the mean" if stage == 0 else f"prediction after {stage} tree" + ("s" if stage > 1 else "") + ", with residuals")
    axes[1, stage].scatter(sketch_x[:, 0], residual, color=PLOT_BLUE, zorder=3)
    axes[1, stage].plot(grid[:, 0], tree.predict(grid), color=PLOT_ORANGE, linewidth=2, drawstyle="steps-mid")
    axes[1, stage].set_title(f"tree {stage + 1}, fitted to the residuals above")
    print(f"stage {stage + 1}: mean absolute residual {np.abs(residual).mean():.4f}")
    # The tree's answer is added after shrinking it by the learning rate, here 0.6.
    prediction = prediction + 0.6 * tree.predict(sketch_x)
axes[0, 0].set_ylabel("response")
axes[1, 0].set_ylabel("residual")
plt.tight_layout()
plt.show()

**The mean absolute residual of the twelve toy rows falls from 0.6346 before the first tree to 0.3168 after it and 0.2062 after two trees.** In the top row each grey arrow runs from the prediction after the earlier trees to a row's response; the tree below it is fitted to exactly those arrow lengths.

__Step 24:__ Draw one gradient boosting stage as a flow: what goes in, what is fitted and what is passed on.

In [ ]:
# Each box is a patch on a blank axis: what goes into one gradient boosting stage, what is fitted and what is passed on. Arrows show the order inside one stage.
stage_boxes = ['prediction F after the earlier trees\n(the mean at the start)', 'pseudo-residuals r:\nthe negative gradient of the loss', 'fit a regression tree h\nto the pseudo-residuals', 'F + learning rate x h:\na shrunk update']
fig, ax = plt.subplots(figsize=(12, 2.8))
for position, label in enumerate(stage_boxes):
    ax.add_patch(FancyBboxPatch((position * 3.1, 0), 2.5, 1.3, boxstyle="round,pad=0.06", facecolor="white", edgecolor=PLOT_ORANGE, linewidth=2))
    ax.text(position * 3.1 + 2.5 / 2, 0.65, label, ha="center", va="center", fontsize=10)
    if position:
        ax.annotate("", xy=(position * 3.1, 0.65), xytext=(position * 3.1 - 0.6, 0.65), arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5))
# The last box feeds the first box of the next stage.
ax.plot([10.55, 10.55, 1.25, 1.25], [-0.08, -0.45, -0.45, -0.2], color="black", linewidth=1.5)
ax.annotate("", xy=(1.25, -0.08), xytext=(1.25, -0.25), arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5))
ax.text((9.3 + 2.5) / 2, -0.75, "the updated prediction is the input of the next stage", ha="center", fontsize=10)
ax.set(xlim=(-0.3, 9.3 + 2.5 + 0.3), ylim=(-1.1, 1.6))
ax.axis("off")
plt.tight_layout()
plt.show()

**One gradient boosting stage takes the prediction after the earlier trees in and passes a shrunk update on.** The pseudo-residuals are computed from that prediction, a tree is fitted to them, and only the learning rate times its answer is added.

__Step 25:__ Compute residuals for six toy responses, fit the first correction tree, and add half its answer.

In [ ]:
toy_response = np.array([2., 3., 2., 8., 9., 8.])
toy_position = np.arange(6).reshape(-1, 1)
toy_initial = np.full(6, toy_response.mean())
toy_residual = toy_response - toy_initial
toy_correction = DecisionTreeRegressor(max_depth=1, random_state=random_state).fit(toy_position, toy_residual)
toy_updated = toy_initial + 0.5 * toy_correction.predict(toy_position)
print(pd.DataFrame({"response": toy_response, "initial": toy_initial, "residual": toy_residual, "correction": toy_correction.predict(toy_position), "updated": toy_updated}).round(3).to_string(index=False))

For squared error, start at the response mean $F_0$ and compute residual $r_i=y_i-F_0$. Fit correction tree $h$ to those residuals, then set $F_1(x)=F_0+\eta h(x)$. Here $i$ indexes rows, $y_i$ is an observed response, $x$ is an input and $\eta$ is the learning rate. For a general loss $L$, the residual is replaced by the pseudo-residual $r_i=-\partial L(y_i,F(x_i))/\partial F(x_i)$, the negative slope of the loss with respect to the prediction $F(x_i)$ of the earlier trees; the half squared error $L=\tfrac{1}{2}(y-F)^2$ gives back $y_i-F(x_i)$.

__Step 26:__ Draw the fitted toy residual tree.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
plot_tree(toy_correction, feature_names=["toy position"], filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title("Gradient boosting: first correction")
plt.tight_layout()
plt.show()

**The correction tree splits the six rows at position 2.5 and answers -3.0 on the left and 3.0 on the right.** Adding half of each answer to the starting mean 5.333 gives updated values of 3.833 and 6.833, one step of the way towards the responses.

Gradient boosting starts from an ordinary model. Here stage 0 is a regular shallow regression tree: the mean response of a row's leaf is the starting prediction. Every later stage computes how wrong the prediction still is for each row, fits a regression tree to that error, and adds a shrunk copy of the tree's answer to the prediction. The tables below show every number for every row, and scikit-learn reproduces them.

__Step 27:__ Fit a regular regression tree to ten toy rows and draw it: its leaf means are the starting prediction.

In [ ]:
# Ten toy rows with one input, the position, and a numeric response.
walk_x = np.arange(10).reshape(-1, 1)
walk_response = np.array([1.0, 1.6, 2.0, 3.1, 3.4, 4.8, 5.0, 6.7, 7.2, 8.0])
# Stage 0 is a regular shallow regression tree fitted to the responses.
walk_start = DecisionTreeRegressor(max_depth=1, random_state=random_state).fit(walk_x, walk_response)
# Its leaf means, the average response in the leaf of each row, are the starting prediction.
walk_start_prediction = walk_start.predict(walk_x)
# Draw the tree: each leaf shows how many rows it holds and their mean response.
fig, ax = plt.subplots(figsize=(7, 3))
plot_tree(walk_start, feature_names=["toy position"], filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title("stage 0: a regular regression tree")
plt.tight_layout()
plt.show()
print("starting prediction of each row:", walk_start_prediction.round(3))

**The stage 0 tree gives the five rows of its left leaf the prediction 2.22 and the five rows of its right leaf 6.34.** These leaf means are the starting prediction that stage 1 starts from.

__Step 28:__ Work out the first gradient boosting stage by hand for every row: prediction, residual, leaf value and update.

In [ ]:
# The learning rate shrinks every tree's answer before it is added.
walk_rate = 0.5
walk_prediction = walk_start_prediction
# Residual: the response minus the prediction, which is minus the slope of the half squared error at the prediction.
walk_residual = walk_response - walk_prediction
# A regression tree is fitted to the residuals, not to the responses.
walk_tree = DecisionTreeRegressor(max_depth=3, random_state=random_state).fit(walk_x, walk_residual)
# For squared error the leaf value is the mean residual of the leaf, so no rescaling is needed.
walk_value = walk_tree.predict(walk_x)
# The new prediction adds the learning rate times the leaf value.
walk_new_prediction = walk_prediction + walk_rate * walk_value
walk_trees = [walk_tree]
walk_tables = [pd.DataFrame({"response": walk_response, "prediction before": walk_prediction, "residual": walk_residual,
    "leaf value": walk_value, "learning rate x leaf value": walk_rate * walk_value, "prediction after": walk_new_prediction})]
display(walk_tables[0].round(3))

**Row 9 has response 8.0 against a starting prediction of 6.34, so its residual is 1.66, its leaf value is 1.66, and after the learning rate of 0.5 its prediction rises from 6.34 to 7.17.** Rows 0 to 2 share one leaf with the value -0.687 and move down together.

The loss of one row is $L=(y-F)^2/2$, where $y$ is the response and $F$ is the prediction. Its slope at $F$ is $F-y$, so the residual $y-F$ is minus the slope: it is positive when the prediction is too low, because moving $F$ up lowers the loss. For squared error the curvature is one, so a leaf value is simply the mean residual of the leaf.

__Step 29:__ Draw the loss of each row against its prediction, the tangent at the prediction before stage 1 and the move made by the stage.

In [ ]:
# Half squared error of one row against its prediction F: L = (y - F)^2 / 2, one parabola for each response.
walk_scores = np.linspace(0, 9, 200)
# The loss of every row before and after the stage, from the predictions of the table above.
walk_loss_before = (walk_response - walk_prediction) ** 2 / 2
walk_loss_after = (walk_response - walk_new_prediction) ** 2 / 2
fig, ax = plt.subplots(figsize=(8, 5))
# For each row: its parabola, a dot at its prediction before the stage, the tangent there, and an arrow to its prediction after the stage.
for row in range(len(walk_response)):
    ax.plot(walk_scores, (walk_response[row] - walk_scores) ** 2 / 2, color=PLOT_BLUE, linewidth=1, alpha=0.35)
    ax.scatter(walk_prediction[row], walk_loss_before[row], color=PLOT_ORANGE, zorder=4)
    ax.plot([walk_prediction[row] - 0.5, walk_prediction[row] + 0.5], [walk_loss_before[row] + 0.5 * walk_residual[row], walk_loss_before[row] - 0.5 * walk_residual[row]], color="gray", linewidth=1)
    ax.annotate("", xy=(walk_new_prediction[row], walk_loss_after[row]), xytext=(walk_prediction[row], walk_loss_before[row]), arrowprops=dict(arrowstyle="->", color=PLOT_ORANGE))
ax.set(xlabel="prediction F", ylabel="loss of the row (half squared error)", ylim=(0, 6), title="stage 1: the tangent slope is minus the residual, the arrow is the update")
plt.tight_layout()
plt.show()
print(f"total loss of the ten rows before stage 1 {walk_loss_before.sum():.4f}, after stage 1 {walk_loss_after.sum():.4f}")

**The total loss of the ten rows falls from 5.9400 to 1.6994 in the first stage.** Row 9, with the largest residual, 1.66, has the steepest tangent and moves furthest, 0.83.

__Step 30:__ Work out the second and third stages the same way.

In [ ]:
# Stages 2 and 3 repeat stage 1; each starts from the prediction the previous stage ended with.
for stage in tqdm((2, 3), desc='stage', leave=True, position=0):
    walk_prediction = walk_new_prediction
    walk_residual = walk_response - walk_prediction
    walk_tree = DecisionTreeRegressor(max_depth=3, random_state=random_state).fit(walk_x, walk_residual)
    walk_value = walk_tree.predict(walk_x)
    walk_new_prediction = walk_prediction + walk_rate * walk_value
    walk_trees.append(walk_tree)
    walk_tables.append(pd.DataFrame({"response": walk_response, "prediction before": walk_prediction, "residual": walk_residual,
        "leaf value": walk_value, "learning rate x leaf value": walk_rate * walk_value, "prediction after": walk_new_prediction}))
    print(f"stage {stage}")
    display(walk_tables[-1].round(3))

**Row 0, with response 1.0, goes from the starting prediction 2.22 to 1.877, 1.438 and 1.232 after stages 1, 2 and 3, so each stage moves it towards its response.**

__Step 31:__ Draw the regression tree fitted at each stage.

In [ ]:
# One full-width figure per stage: the regression tree fitted to that stage's residuals. Its boxes show the mean pseudo-residual of each leaf.
for stage, tree in enumerate(walk_trees, start=1):
    fig, axis = plt.subplots(figsize=(12, 5))
    plot_tree(tree, feature_names=["toy position"], filled=True, rounded=True, precision=2, fontsize=9, ax=axis)
    axis.set_title(f"stage {stage}: regression tree on the residuals")
    plt.tight_layout()
    plt.show()

**Each figure shows the regression tree of one stage, and its leaf boxes show the mean residual of the rows in the leaf.** For squared error that mean is the leaf value used in the update.

__Step 32:__ Check each hand stage against scikit-learn's `GradientBoostingRegressor`, started from the same stage 0 tree.

In [ ]:
# scikit-learn started from the same stage 0 tree, with the same learning rate and tree depth, must reach the same predictions.
walk_library = GradientBoostingRegressor(init=walk_start, n_estimators=3, learning_rate=walk_rate, max_depth=3, random_state=random_state).fit(walk_x, walk_response)
walk_staged = list(walk_library.staged_predict(walk_x))
# Compare the prediction after each stage with the hand table, within a stated tolerance of 1e-8.
for stage, (table, library) in enumerate(zip(walk_tables, walk_staged), start=1):
    assert np.allclose(table["prediction after"], library, rtol=0, atol=1e-8), stage
print("the three hand stages equal staged_predict within 1e-8; largest difference",
      max(np.abs(table["prediction after"] - library).max() for table, library in zip(walk_tables, walk_staged)))

**The three hand stages match the staged predictions of scikit-learn's `GradientBoostingRegressor` to within 1e-8, with a largest difference of 0.0.** Starting the library model from the same stage 0 tree is what makes the two start from the same predictions.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html'>sklearn.ensemble.GradientBoostingRegressor(loss='squared_error', learning_rate=0.1, n_estimators=100, subsample=1.0, min_samples_split=2, min_samples_leaf=1, max_depth=3, max_features=None, alpha=0.9, ccp_alpha=0.0, random_state=None)</a>

**Definition:**
Adds one regression tree per stage, fitted to the negative gradient of the loss at the current predictions, and sums the scaled trees into the prediction.

**Main Parameters:**
- `loss`: loss whose gradient each tree fits.
- `learning_rate`, `n_estimators`: tree multiplier and number of trees.
- `subsample`: fraction of rows drawn for each tree.
- `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`, `ccp_alpha`: growth controls of each tree.
- `alpha`: quantile of the Huber and quantile losses.
- `random_state`: seed of row and column draws.

**Main Attributes (after fitting):**
- `estimators_`: the fitted regression trees.
- `init_`: the model that gives the starting prediction.
- `train_score_`: the training loss after each stage.

**Main Methods:**
- `fit(X, y)`: add the trees in sequence.
- `predict(X)`: the summed prediction.
- `staged_predict(X)`: the prediction after each stage.
</div>

__Step 33:__ Score the GradientBoostingRegressor reference on the ten prepared training folds.

In [ ]:
gb_reference = GradientBoostingRegressor(random_state=random_state)
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(gb_reference).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
gb_default_train = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
gb_default_validation = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]

__Step 34:__ Fit the GradientBoostingRegressor reference on all the training rows.

In [ ]:
gb_default = clone(gb_reference).fit(Z_train, y_train)
gb_default_configuration = " ".join(str(gb_reference).split())
gb_complete_rows = []

__Step 35:__ Log the GradientBoostingRegressor reference.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": gb_default_configuration, "train": gb_default_train, "validation": gb_default_validation,
    "model": gb_default},
    Z_test, y_test, "untuned", metric=mean_absolute_error, label="MAE")

# <font color='#E8800A'>Gradient boosting settings</font> <a class="anchor" id="gb_settings"></a>
[Back to TOC](#toc)

Each sweep changes one gradient boosting setting from the default model and scores every value on the same ten prepared training folds. The section closes on the best configuration actually evaluated; the rate and stage-count interaction is measured separately. The other tree settings, `min_samples_leaf`, `min_samples_split`, `max_features` and `ccp_alpha`, were taught with decision trees and stay at their reference values here.

| Parameter | Meaning |
| --- | --- |
| `n_estimators` | Number of regression trees. |
| `learning_rate` | Multiplier on each tree's answer. |
| `max_depth` | Splits allowed on a path of a stage tree. |
| `subsample` | Fraction of rows drawn for each tree. |
| `loss` | Loss whose negative gradient each tree fits. |
| `alpha` | Huber threshold quantile or quantile level (regression only). |

[Parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html)

__Step 36:__ Change only gradient boosting `n_estimators` on the fixed training folds, with the sweep list `[1, 5, 10, 30, 50, 70, 90, 110, 130, 150]`.

### `gradient boosting: n_estimators`

Each stage adds a regression tree to the current prediction; the default is `100`. More stages are expected to reduce training loss and increase runtime, but validation may stop improving. The learning rate controls the size of each correction. The candidate list is `[1, 5, 10, 30, 50, 70, 90, 110, 130, 150]`.

In [ ]:
# The stage counts to report; one fit per fold at the largest count gives the prediction after every smaller count as well.
gb_n_estimators_values = [1, 5, 10, 30, 50, 70, 90, 110, 130, 150]
gb_n_estimators_rows = []
gb_stage_count = max(gb_n_estimators_values)
# Arrays of shape folds by stages: the training and validation score of each fold after each stage.
gb_train_by_fold = np.zeros((len(prepared), gb_stage_count))
gb_validation_by_fold = np.zeros((len(prepared), gb_stage_count))
# The ten fold fits run at the same time.
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(gb_reference).set_params(n_estimators=gb_stage_count).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
# `staged_predict` yields the prediction after stage 1, 2, 3 and so on: score each stage on the fitted rows and on the validation rows.
for position, (fold, fitted) in tqdm(list(enumerate(zip(prepared, fits))), desc="GradientBoostingRegressor staged scores", unit="fold"):
    on_fit = list(fitted.staged_predict(fold["X_fit"]))
    on_score = list(fitted.staged_predict(fold["X_score"]))
    for stage in range(gb_stage_count):
        used = min(stage, len(on_fit) - 1)
        gb_train_by_fold[position, stage] = mean_absolute_error(fold["y_fit"], on_fit[used])
        gb_validation_by_fold[position, stage] = mean_absolute_error(fold["y_score"], on_score[used])
    # Keep the first fold's staged answers for the check below.
    if position == 0:
        first_staged = on_score
# Check: a fresh fit with fewer stages must predict exactly like the early part of the long fit, at each count before the last.
for count in tqdm(gb_n_estimators_values[:-1], desc='gb n estimators values[:-1]', leave=True, position=0):
    fresh = clone(gb_reference).set_params(n_estimators=count).fit(prepared[0]["X_fit"], prepared[0]["y_fit"])
    assert np.allclose(fresh.predict(prepared[0]["X_score"]), first_staged[min(count, len(first_staged)) - 1]), count
print("staged predictions equal fresh fits at", gb_n_estimators_values[:-1], "stages")
# One row per reported count: its fold scores come from the staged arrays, and its all-rows fit is scored on the sealed test rows.
for value in tqdm(gb_n_estimators_values, desc='GradientBoostingRegressor.n_estimators', unit='value', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(gb_reference).set_params(n_estimators=value)
    train_scores = list(gb_train_by_fold[:, value - 1])
    validation_scores = list(gb_validation_by_fold[:, value - 1])
    deployed = clone(candidate).fit(Z_train, y_train)
    gb_n_estimators_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per reported count, with the mean fold scores, the test score and the seconds spent on its all-rows fit.
gb_n_estimators_sweep = pd.DataFrame(gb_n_estimators_rows)
gb_complete_rows.extend(gb_n_estimators_rows)
gb_n_estimators_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
gb_n_estimators_winner = gb_n_estimators_sweep.loc[gb_n_estimators_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": gb_n_estimators_winner["configuration"], "train": gb_n_estimators_winner["train"],
    "validation": gb_n_estimators_winner["validation"], "model": gb_n_estimators_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**Validation MAE falls from 6,741.07 EUR with one stage to 2,808.50 at 30 stages and 2,401.41 at 150, while training MAE falls from 6,738.36 to 1,943.83.** Both lines in the figure are still falling at 150 stages, and the validation line stays above the training line at every count.

__Step 37:__ Plot training and validation MAE against the number of gradient boosting stages.

In [ ]:
# Lines: the mean score of the ten folds after every stage. Dots: the counts reported in the table above.
fig, ax = plt.subplots(figsize=(7, 5))
gb_stages = np.arange(1, gb_stage_count + 1)
ax.plot(gb_stages, gb_train_by_fold.mean(axis=0), color=PLOT_ORANGE, linewidth=2, label="training (ten folds)")
ax.plot(gb_stages, gb_validation_by_fold.mean(axis=0), color=PLOT_BLUE, linewidth=2, label="validation (ten folds)")
ax.scatter(gb_n_estimators_sweep["setting"], gb_n_estimators_sweep["train_MAE"], color=PLOT_ORANGE, zorder=3)
ax.scatter(gb_n_estimators_sweep["setting"], gb_n_estimators_sweep["cv_MAE"], color=PLOT_BLUE, zorder=3)
ax.set(xlabel="n_estimators", ylabel="MAE", title="GradientBoostingRegressor: train against validation")
ax.legend()
plt.tight_layout()
plt.show()

__Step 38:__ Change only gradient boosting `learning_rate` on the fixed training folds, with the sweep list `[0.03, 0.1, 0.3]`.

### `gradient boosting: learning_rate`

The learning rate multiplies each new tree's correction; the default is `0.1`. A smaller rate is expected to need more stages, while a larger rate can reach a useful fit sooner and then overfit. Validation, rather than an equal rate-times-stage product, decides the useful combination. The candidate list is `[0.03, 0.1, 0.3]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
gb_learning_rate_values = [0.03, 0.1, 0.3]
# One row per value will hold its scores, its seconds and its fitted model.
gb_learning_rate_rows = []
for value in tqdm(gb_learning_rate_values, desc="GradientBoostingRegressor.learning_rate", unit="value"):
    started = time.perf_counter()
    candidate = clone(gb_reference).set_params(learning_rate=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    gb_learning_rate_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
gb_learning_rate_sweep = pd.DataFrame(gb_learning_rate_rows)
gb_complete_rows.extend(gb_learning_rate_rows)
gb_learning_rate_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
gb_learning_rate_winner = gb_learning_rate_sweep.loc[gb_learning_rate_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": gb_learning_rate_winner["configuration"], "train": gb_learning_rate_winner["train"],
    "validation": gb_learning_rate_winner["validation"], "model": gb_learning_rate_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**A larger step wins within the three tried: validation MAE is 2,837.98 EUR at 0.03, 2,464.43 at 0.1 and 2,362.40 at 0.3.** Training MAE falls from 2,608.83 EUR to 1,682.43 across the same rates.

__Step 39:__ Change only gradient boosting `max_depth` on the fixed training folds, with the sweep list `[1, 3, 5]`.

### `gradient boosting: max_depth`

Depth limits the number of splits along each tree path and therefore which feature interactions one stage can model. The default is `3`. Deeper trees are expected to fit residual structure more closely, increasing both runtime and the risk of fitting noise. The candidate list is `[1, 3, 5]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
gb_max_depth_values = [1, 3, 5]
# One row per value will hold its scores, its seconds and its fitted model.
gb_max_depth_rows = []
for value in tqdm(gb_max_depth_values, desc="GradientBoostingRegressor.max_depth", unit="value"):
    started = time.perf_counter()
    candidate = clone(gb_reference).set_params(max_depth=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    gb_max_depth_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
gb_max_depth_sweep = pd.DataFrame(gb_max_depth_rows)
gb_complete_rows.extend(gb_max_depth_rows)
gb_max_depth_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
gb_max_depth_winner = gb_max_depth_sweep.loc[gb_max_depth_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": gb_max_depth_winner["configuration"], "train": gb_max_depth_winner["train"],
    "validation": gb_max_depth_winner["validation"], "model": gb_max_depth_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**Validation MAE is 3,039.48 EUR at depth 1, 2,464.43 at 3 and 2,284.84 at 5.** Training MAE falls from 2,945.44 EUR to 1,499.24, so depth 5 still lowers validation error while fitting the training folds much more closely.

__Step 40:__ Change only gradient boosting `subsample` on the fixed training folds, with the sweep list `[0.6, 0.8, 1.0]`.

### `gradient boosting: subsample`

Each stage samples this fraction of training rows without replacement; the default `1.0` uses every row. A smaller fraction is expected to lower runtime and make the stage trees differ. It can reduce variance while increasing bias. The candidate list is `[0.6, 0.8, 1.0]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
gb_subsample_values = [0.6, 0.8, 1.0]
# One row per value will hold its scores, its seconds and its fitted model.
gb_subsample_rows = []
for value in tqdm(gb_subsample_values, desc="GradientBoostingRegressor.subsample", unit="value"):
    started = time.perf_counter()
    candidate = clone(gb_reference).set_params(subsample=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    gb_subsample_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
gb_subsample_sweep = pd.DataFrame(gb_subsample_rows)
gb_complete_rows.extend(gb_subsample_rows)
gb_subsample_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
gb_subsample_winner = gb_subsample_sweep.loc[gb_subsample_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": gb_subsample_winner["configuration"], "train": gb_subsample_winner["train"],
    "validation": gb_subsample_winner["validation"], "model": gb_subsample_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**Drawing fewer rows per stage does not help: validation MAE is 2,503.04 EUR at 0.6, 2,470.41 at 0.8 and 2,464.43 at 1.0.**

__Step 41:__ Change only gradient boosting `loss` on the fixed training folds, with the sweep list `['squared_error', 'absolute_error', 'huber', 'quantile']`.

### `gradient boosting: loss`

The default `squared_error` targets the conditional mean; `absolute_error` targets the median. Huber loss limits the influence of large residuals, while quantile loss targets the level set by alpha. Robust losses are expected to give extreme target values less influence. The candidate list is `['squared_error', 'absolute_error', 'huber', 'quantile']`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
gb_loss_values = ['squared_error', 'absolute_error', 'huber', 'quantile']
# One row per value will hold its scores, its seconds and its fitted model.
gb_loss_rows = []
for value in tqdm(gb_loss_values, desc="GradientBoostingRegressor.loss", unit="value"):
    started = time.perf_counter()
    candidate = clone(gb_reference).set_params(loss=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    gb_loss_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
gb_loss_sweep = pd.DataFrame(gb_loss_rows)
gb_complete_rows.extend(gb_loss_rows)
gb_loss_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
gb_loss_winner = gb_loss_sweep.loc[gb_loss_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": gb_loss_winner["configuration"], "train": gb_loss_winner["train"],
    "validation": gb_loss_winner["validation"], "model": gb_loss_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**The Huber loss scores the best validation MAE, 2,370.53 EUR, ahead of absolute error at 2,419.74 and squared error at 2,464.43, while the quantile loss scores 4,250.48.**

__Step 42:__ Change only gradient boosting `alpha` on the fixed training folds, with the sweep list `[0.5, 0.8, 0.95]`.

### `gradient boosting: alpha`

For Huber loss, alpha sets the residual quantile where squared error gives way to absolute error. The default is `0.9`; this conditional sweep freezes `loss="huber"` before varying alpha. A higher value is expected to treat more residuals with squared error and give large errors more influence. The candidate list is `[0.5, 0.8, 0.95]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
gb_alpha_reference = clone(gb_reference).set_params(loss="huber")
gb_alpha_values = [0.5, 0.8, 0.95]
# One row per value will hold its scores, its seconds and its fitted model.
gb_alpha_rows = []
for value in tqdm(gb_alpha_values, desc="GradientBoostingRegressor.alpha", unit="value"):
    started = time.perf_counter()
    candidate = clone(gb_alpha_reference).set_params(alpha=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    gb_alpha_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
gb_alpha_sweep = pd.DataFrame(gb_alpha_rows)
gb_complete_rows.extend(gb_alpha_rows)
gb_alpha_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
gb_alpha_winner = gb_alpha_sweep.loc[gb_alpha_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": gb_alpha_winner["configuration"], "train": gb_alpha_winner["train"],
    "validation": gb_alpha_winner["validation"], "model": gb_alpha_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**With the Huber loss, an alpha of 0.8 scores the best validation MAE, 2,365.19 EUR, against 2,386.51 at 0.5 and 2,397.87 at 0.95.**

__Step 43:__ Plot training and validation MAE for every other swept gradient boosting setting, one panel each.

In [ ]:
# The sweep tables of this family, each with its list of swept values for the axis labels.
gb_settings_tables = {"learning_rate": (gb_learning_rate_sweep, gb_learning_rate_values), "max_depth": (gb_max_depth_sweep, gb_max_depth_values), "subsample": (gb_subsample_sweep, gb_subsample_values), "loss": (gb_loss_sweep, gb_loss_values), "alpha": (gb_alpha_sweep, gb_alpha_values)}
# One panel per setting: training and validation score at every swept value, side by side.
fig, axes = plt.subplots(2, 3, figsize=(12, 7.2), squeeze=False)
for axis, (setting, (table, values)) in zip(axes.ravel(), gb_settings_tables.items()):
    positions = np.arange(len(table))
    axis.plot(positions, table["train_MAE"], marker="o", color=PLOT_ORANGE, label="training")
    axis.plot(positions, table["cv_MAE"], marker="o", color=PLOT_BLUE, label="validation")
    axis.set(title=setting, xticks=positions, xticklabels=[str(value) for value in values], ylabel="MAE")
    axis.tick_params(axis="x", labelrotation=20)
axes.ravel()[0].legend()
# Hide the panels left over in the last row.
for axis in axes.ravel()[len(gb_settings_tables):]:
    axis.set_visible(False)
plt.tight_layout()
plt.show()

**Validation MAE stays between 2,284.84 EUR and 2,503.04 in the five panels except for a depth of 1 at 3,039.48, a learning rate of 0.03 at 2,837.98 and the quantile loss at 4,250.48.** Training MAE is lowest, 1,499.24, at depth 5.

__Step 44:__ Draw the validation MAE of gradient boosting at every stage for four learning rates.

### `gradient boosting: learning rate and stage count`

Smaller corrections often need more stages, but noise, overfitting and early termination can break that pattern. The first study plots actual fitted stages against validation score at each learning rate, with an explicit larger stage budget. The second enables a stopping criterion; validation determines whether its chosen count helps.

In [ ]:
# Four learning rates, each with its own staged validation curve (mean over the ten folds).
gb_curve_rates = [0.03, 0.1, 0.3, 1.0]
gb_rate_curves = {}
# Fit every fold once per rate at the largest stage count, then score the prediction after each stage: one fit gives the whole curve.
for rate in tqdm(gb_curve_rates, desc="GradientBoostingRegressor learning rates", unit="rate"):
    fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(gb_reference).set_params(learning_rate=rate, n_estimators=300).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
    gb_rate_curves[rate] = np.mean([[mean_absolute_error(fold["y_score"], on_score) for on_score in fitted.staged_predict(fold["X_score"])] for fold, fitted in zip(prepared, fits)], axis=0)
# Draw the curves; the dot marks the stage with the best validation score of each rate.
fig, ax = plt.subplots(figsize=(7, 5))
for rate, curve in gb_rate_curves.items():
    ax.plot(np.arange(1, len(curve) + 1), curve, linewidth=2, label=f"learning rate {rate}")
    ax.scatter(np.argmin(curve) + 1, curve.min(), zorder=3)
ax.set(xlabel="n_estimators", ylabel="validation MAE", title="GradientBoostingRegressor: validation by stage at four learning rates")
ax.legend()
plt.tight_layout()
plt.show()
# Table of the same curves: the best stage, the stage where each curve first comes close to its best, and that best score.
pd.DataFrame({"best stage": [int(np.argmin(curve)) + 1 for curve in gb_rate_curves.values()],
    "stages to come within 1 percent of the best": [int(np.argmax(curve <= 1.01 * curve.min())) + 1 for curve in gb_rate_curves.values()],
    "best validation MAE": [curve.min() for curve in gb_rate_curves.values()]}, index=pd.Index(gb_curve_rates, name="learning_rate")).round(4)

**A rate of 0.03 is still improving at stage 300 with validation MAE 2,486.66 EUR, a rate of 0.1 reaches 2,316.99 at stage 298 and a rate of 0.3 reaches 2,340.19 at stage 155, while a rate of 1.0 is best at stage 32 with 2,696.31.** Within 1 percent of its best, the rate 0.03 needs 250 stages, the rate 0.1 needs 241 and the rate 0.3 needs 98, so shrinkage buys more stages for the same fit.

__Step 45:__ Stop each gradient boosting fit with its convergence criterion and count the stages it used.

In [ ]:
# With a convergence criterion each fit stops by itself, so the learning rate decides how many stages are fitted.
gb_stop_rows = []
for rate in tqdm(gb_curve_rates, desc="GradientBoostingRegressor early stopping", unit="rate"):
    started = time.perf_counter()
    fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(gb_reference).set_params(learning_rate=rate, n_estimators=1000, n_iter_no_change=10, validation_fraction=0.1).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
    gb_stop_rows.append({"learning_rate": rate, "stages fitted": np.mean([fitted.n_estimators_ for fitted in fits]),
        "validation MAE": np.mean([mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]),
        "seconds": time.perf_counter() - started})
pd.DataFrame(gb_stop_rows).set_index("learning_rate").round(4)

**With the criterion on, the fit stops after 336.6 stages on average over the folds at a rate of 0.03, 145.2 at 0.1, 64.8 at 0.3 and 40.7 at 1.0.** The validation MAE is 2,533.53 EUR, 2,478.31, 2,453.30 and 2,724.42, so the learning rate decides when training stops and how far it gets.

__Step 46:__ Select the best tested gradient boosting configuration from the existing sweep rows by validation score.

In [ ]:
gb_complete = pd.DataFrame(gb_complete_rows)
gb_best = gb_complete.loc[gb_complete["cv_MAE"].idxmin()]
gb_best_parameters = gb_best["parameters"]
gb_best_configuration = clone(gb_reference).set_params(**gb_best_parameters)
print("best tested gradient boosting configuration:", gb_best["configuration"])

**The best complete gradient boosting configuration measured uses depth 5, with validation MAE 2,284.84 EUR.**

# <font color='#E8800A'>Histogram gradient boosting</font> <a class="anchor" id="hist"></a>
[Back to TOC](#toc)

Histogram boosting groups numeric values into bins before considering splits. It retains sequential loss corrections while reducing the number of candidate thresholds. [Histogram boosting guide](https://scikit-learn.org/stable/modules/ensemble.html#histogram-based-gradient-boosting)

__Step 47:__ Group six toy values into three bins and count the rows offered to each bin.

In [ ]:
toy_values = np.array([1., 2., 3., 7., 8., 9.])
toy_edges = np.array([3.5, 7.5])
toy_bins = np.searchsorted(toy_edges, toy_values)
print(pd.DataFrame({"value": toy_values, "bin": toy_bins}).to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(np.arange(3), np.bincount(toy_bins, minlength=3), color=PLOT_BLUE)
ax.set(xlabel="Bin", ylabel="Training rows", title="Three toy histogram bins")
plt.tight_layout()
plt.show()

**The values 1, 2 and 3 fall in bin 0, the value 7 in bin 1 and the values 8 and 9 in bin 2, so the three bins hold 3, 1 and 2 rows.**

__Step 48:__ Draw the toy values on a line with the candidate cuts of an exact tree and of histogram boosting.

In [ ]:
# Top: an exact tree may cut halfway between any two neighbouring values. Bottom: histogram boosting may cut only at the bin edges.
bin_midpoints = (toy_values[:-1] + toy_values[1:]) / 2
bin_colours = [PLOT_BLUE, PLOT_ORANGE, "#4C4C4C"]
fig, axes = plt.subplots(2, 1, figsize=(9, 3.8), sharex=True)
for axis, thresholds, title in zip(axes, [bin_midpoints, toy_edges], ["exact tree", "histogram boosting"]):
    axis.scatter(toy_values, np.zeros(len(toy_values)), c=[bin_colours[bin_index] for bin_index in toy_bins], s=140, zorder=3)
    for threshold in thresholds:
        axis.axvline(threshold, color="gray", linestyle="--")
    axis.set(title=f"{title}: {len(thresholds)} candidate cuts", yticks=[])
    print(f"{title}: {len(thresholds)} candidate cuts at {np.round(thresholds, 1).tolist()}")
# Shade the bins on the bottom panel: every value inside one shade is treated alike when a cut is chosen.
for (left, right), shade in zip(zip(np.r_[toy_values.min() - 0.5, toy_edges], np.r_[toy_edges, toy_values.max() + 0.5]), bin_colours):
    axes[1].axvspan(left, right, color=shade, alpha=0.15)
axes[1].set_xlabel("raw value")
plt.tight_layout()
plt.show()

**Binning cuts the exact tree's candidate thresholds down to the bin edges.** Every value inside one bin is treated alike when a cut is chosen, which makes the search for a split cheaper; the stages then proceed as in gradient boosting.

__Step 49:__ Draw one histogram boosting stage as a flow: binned input, then a gradient boosting stage.

In [ ]:
# Each box is a patch on a blank axis: the binned input of histogram boosting and one stage that then proceeds as in gradient boosting. Arrows show the order inside one stage.
stage_boxes = ['columns cut into bins\n(once, before stage 1)', 'gradient and hessian\nof the loss per row', 'tree grown leaf by leaf\non the bin sums', 'F + learning rate x h:\na shrunk update']
fig, ax = plt.subplots(figsize=(12, 2.8))
for position, label in enumerate(stage_boxes):
    ax.add_patch(FancyBboxPatch((position * 3.1, 0), 2.5, 1.3, boxstyle="round,pad=0.06", facecolor="white", edgecolor="#4C4C4C", linewidth=2))
    ax.text(position * 3.1 + 2.5 / 2, 0.65, label, ha="center", va="center", fontsize=10)
    if position:
        ax.annotate("", xy=(position * 3.1, 0.65), xytext=(position * 3.1 - 0.6, 0.65), arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5))
# The last box feeds the first box of the next stage.
ax.plot([10.55, 10.55, 1.25, 1.25], [-0.08, -0.45, -0.45, -0.2], color="black", linewidth=1.5)
ax.annotate("", xy=(1.25, -0.08), xytext=(1.25, -0.25), arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5))
ax.text((9.3 + 2.5) / 2, -0.75, "the updated prediction is the input of the next stage; the bins are not recomputed", ha="center", fontsize=10)
ax.set(xlim=(-0.3, 9.3 + 2.5 + 0.3), ylim=(-1.1, 1.6))
ax.axis("off")
plt.tight_layout()
plt.show()

**Histogram boosting bins the columns once, and every stage after that is a gradient boosting stage.** The tree is grown leaf by leaf on sums of the gradient and hessian within the bins, and its shrunk answer updates the prediction.

__Step 50:__ Draw the three loops side by side: what goes in, what is fitted and what is passed on.

In [ ]:
# Rows: the three algorithms. Columns: what goes into a stage, what is fitted, and what is passed on.
loops = [("AdaBoost", PLOT_BLUE, ["row weights", "a learner fitted\nto the weighted rows", "new row weights and\na stage weight"]),
         ("gradient boosting", PLOT_ORANGE, ["prediction after the\nearlier trees", "a tree fitted to the\npseudo-residuals", "prediction plus a\nshrunk tree"]),
         ("histogram boosting", "#4C4C4C", ["binned columns, then the prediction\nafter the earlier trees", "a tree grown leaf by leaf\non bin sums", "prediction plus a\nshrunk tree"])]
fig, ax = plt.subplots(figsize=(12, 4.6))
for column, header in enumerate(["goes in", "is fitted", "is passed on"]):
    ax.text(column * 4.4 + 1.7, 4.95, header, ha="center", fontsize=11, fontweight="bold")
for row, (name, colour, labels) in enumerate(loops):
    height = (2 - row) * 1.65
    ax.text(-0.3, height + 0.6, name, ha="right", va="center", fontsize=11, fontweight="bold")
    for column, label in enumerate(labels):
        ax.add_patch(FancyBboxPatch((column * 4.4, height), 3.4, 1.2, boxstyle="round,pad=0.06", facecolor="white", edgecolor=colour, linewidth=2))
        ax.text(column * 4.4 + 1.7, height + 0.6, label, ha="center", va="center", fontsize=10)
        if column:
            ax.annotate("", xy=(column * 4.4, height + 0.6), xytext=(column * 4.4 - 1.0, height + 0.6), arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5))
ax.set(xlim=(-3.4, 12.8), ylim=(-0.2, 5.3))
ax.axis("off")
plt.tight_layout()
plt.show()

**Read the rows from left to right and compare what each loop passes on.** AdaBoost passes row weights, gradient boosting passes a prediction whose residuals the next tree fits, and histogram boosting passes the same prediction while working on binned columns.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html'>sklearn.ensemble.HistGradientBoostingRegressor(loss='squared_error', quantile=None, learning_rate=0.1, max_iter=100, max_leaf_nodes=31, max_depth=None, min_samples_leaf=20, l2_regularization=0.0, max_features=1.0, max_bins=255, early_stopping='auto', random_state=None)</a>

**Definition:**
Bins every numeric column into at most 255 integer bins, then adds one leaf-wise grown tree per iteration, fitted to the gradient and hessian of the loss.

**Main Parameters:**
- `max_iter`, `learning_rate`: number of iterations and leaf-value multiplier.
- `max_leaf_nodes`, `max_depth`, `min_samples_leaf`: tree size controls.
- `l2_regularization`: penalty that shrinks leaf values.
- `max_bins`, `max_features`: threshold resolution and columns per split.
- `loss`, `quantile`: loss to reduce and the level of the quantile loss.
- `early_stopping`: stops adding iterations when a validation score stalls.

**Main Attributes (after fitting):**
- `n_iter_`: iterations actually fitted.
- `train_score_`: the training score after each iteration when scoring is on.
- `n_trees_per_iteration_`: trees added per iteration.

**Main Methods:**
- `fit(X, y)`: bin the columns and add the trees.
- `predict(X)`: the summed prediction.
- `staged_predict(X)`: the prediction after each iteration.
</div>

__Step 51:__ Score the HistGradientBoostingRegressor reference on the ten prepared training folds.

In [ ]:
hist_reference = HistGradientBoostingRegressor(random_state=random_state)
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(hist_reference).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
hist_default_train = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
hist_default_validation = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]

__Step 52:__ Fit the HistGradientBoostingRegressor reference on all the training rows.

In [ ]:
hist_default = clone(hist_reference).fit(Z_train, y_train)
hist_default_configuration = " ".join(str(hist_reference).split())
hist_complete_rows = []

__Step 53:__ Log the HistGradientBoostingRegressor reference.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": hist_default_configuration, "train": hist_default_train, "validation": hist_default_validation,
    "model": hist_default},
    Z_test, y_test, "untuned", metric=mean_absolute_error, label="MAE")

# <font color='#E8800A'>Histogram boosting settings</font> <a class="anchor" id="hist_settings"></a>
[Back to TOC](#toc)

Each sweep changes one histogram boosting setting from the reference, whose early stopping is off, and scores every value on the same ten prepared training folds. The section closes on the best configuration actually evaluated; interactions need the later joint search. The other tree settings, `max_depth`, `min_samples_leaf` and `max_features`, were taught with decision trees and stay at their reference values here.

| Parameter | Meaning |
| --- | --- |
| `max_iter` | Number of boosting stages. |
| `learning_rate` | Multiplier on each tree's leaf values. |
| `max_leaf_nodes` | Leaves allowed in a tree. |
| `l2_regularization` | Penalty that shrinks leaf values. |
| `max_bins` | Bins per numeric column. |
| `loss` | Loss each tree reduces (regression only). |

[Parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html)

__Step 54:__ Change only histogram boosting `max_iter` on the fixed training folds, with the sweep list `[1, 5, 10, 30, 50, 70, 90, 110, 130, 150]`.

### `histogram boosting: max_iter`

One iteration adds a tree; the default is `100`. The default `early_stopping="auto"` activates only above 10000 training rows, so it stays off for these smaller folds. More iterations are expected to fit training rows more closely, with validation deciding whether the extra cost helps. The candidate list is `[1, 5, 10, 30, 50, 70, 90, 110, 130, 150]`.

In [ ]:
# The stage counts to report; one fit per fold at the largest count gives the prediction after every smaller count as well.
hist_max_iter_values = [1, 5, 10, 30, 50, 70, 90, 110, 130, 150]
hist_max_iter_rows = []
hist_stage_count = max(hist_max_iter_values)
# Arrays of shape folds by stages: the training and validation score of each fold after each stage.
hist_train_by_fold = np.zeros((len(prepared), hist_stage_count))
hist_validation_by_fold = np.zeros((len(prepared), hist_stage_count))
# The ten fold fits run at the same time.
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(hist_reference).set_params(max_iter=hist_stage_count).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
# `staged_predict` yields the prediction after stage 1, 2, 3 and so on: score each stage on the fitted rows and on the validation rows.
for position, (fold, fitted) in tqdm(list(enumerate(zip(prepared, fits))), desc="HistGradientBoostingRegressor staged scores", unit="fold"):
    on_fit = list(fitted.staged_predict(fold["X_fit"]))
    on_score = list(fitted.staged_predict(fold["X_score"]))
    for stage in range(hist_stage_count):
        used = min(stage, len(on_fit) - 1)
        hist_train_by_fold[position, stage] = mean_absolute_error(fold["y_fit"], on_fit[used])
        hist_validation_by_fold[position, stage] = mean_absolute_error(fold["y_score"], on_score[used])
    # Keep the first fold's staged answers for the check below.
    if position == 0:
        first_staged = on_score
# Check: a fresh fit with fewer stages must predict exactly like the early part of the long fit, at each count before the last.
for count in tqdm(hist_max_iter_values[:-1], desc='hist max iter values[:-1]', leave=True, position=0):
    fresh = clone(hist_reference).set_params(max_iter=count).fit(prepared[0]["X_fit"], prepared[0]["y_fit"])
    assert np.allclose(fresh.predict(prepared[0]["X_score"]), first_staged[min(count, len(first_staged)) - 1]), count
print("staged predictions equal fresh fits at", hist_max_iter_values[:-1], "stages")
# One row per reported count: its fold scores come from the staged arrays, and its all-rows fit is scored on the sealed test rows.
for value in tqdm(hist_max_iter_values, desc='HistGradientBoostingRegressor.max_iter', unit='value', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(hist_reference).set_params(max_iter=value)
    train_scores = list(hist_train_by_fold[:, value - 1])
    validation_scores = list(hist_validation_by_fold[:, value - 1])
    deployed = clone(candidate).fit(Z_train, y_train)
    hist_max_iter_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per reported count, with the mean fold scores, the test score and the seconds spent on its all-rows fit.
hist_max_iter_sweep = pd.DataFrame(hist_max_iter_rows)
hist_complete_rows.extend(hist_max_iter_rows)
hist_max_iter_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
hist_max_iter_winner = hist_max_iter_sweep.loc[hist_max_iter_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": hist_max_iter_winner["configuration"], "train": hist_max_iter_winner["train"],
    "validation": hist_max_iter_winner["validation"], "model": hist_max_iter_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**Validation MAE falls from 6,635.29 EUR with one stage to 2,417.65 at 30 stages and reaches its lowest, 2,278.67, at 90 stages, then rises slightly to 2,286.13 at 150, while training MAE keeps falling from 6,620.28 to 1,380.27.** The validation line flattens while the training line keeps falling, so the gap between them widens with every added stage.

__Step 55:__ Plot training and validation MAE against the number of histogram boosting stages.

In [ ]:
# Lines: the mean score of the ten folds after every stage. Dots: the counts reported in the table above.
fig, ax = plt.subplots(figsize=(7, 5))
hist_stages = np.arange(1, hist_stage_count + 1)
ax.plot(hist_stages, hist_train_by_fold.mean(axis=0), color=PLOT_ORANGE, linewidth=2, label="training (ten folds)")
ax.plot(hist_stages, hist_validation_by_fold.mean(axis=0), color=PLOT_BLUE, linewidth=2, label="validation (ten folds)")
ax.scatter(hist_max_iter_sweep["setting"], hist_max_iter_sweep["train_MAE"], color=PLOT_ORANGE, zorder=3)
ax.scatter(hist_max_iter_sweep["setting"], hist_max_iter_sweep["cv_MAE"], color=PLOT_BLUE, zorder=3)
ax.set(xlabel="max_iter", ylabel="MAE", title="HistGradientBoostingRegressor: train against validation")
ax.legend()
plt.tight_layout()
plt.show()

__Step 56:__ Change only histogram boosting `learning_rate` on the fixed training folds, with the sweep list `[0.03, 0.1, 0.3]`.

### `histogram boosting: learning_rate`

The learning rate scales each new tree's leaf values; the default is `0.1`. A smaller rate is expected to need more iterations, while a larger rate reaches a close training fit sooner. Neither guarantees better validation error. The candidate list is `[0.03, 0.1, 0.3]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
hist_learning_rate_values = [0.03, 0.1, 0.3]
# One row per value will hold its scores, its seconds and its fitted model.
hist_learning_rate_rows = []
for value in tqdm(hist_learning_rate_values, desc="HistGradientBoostingRegressor.learning_rate", unit="value"):
    started = time.perf_counter()
    candidate = clone(hist_reference).set_params(learning_rate=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    hist_learning_rate_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
hist_learning_rate_sweep = pd.DataFrame(hist_learning_rate_rows)
hist_complete_rows.extend(hist_learning_rate_rows)
hist_learning_rate_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
hist_learning_rate_winner = hist_learning_rate_sweep.loc[hist_learning_rate_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": hist_learning_rate_winner["configuration"], "train": hist_learning_rate_winner["train"],
    "validation": hist_learning_rate_winner["validation"], "model": hist_learning_rate_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**A rate of 0.1 scores the best validation MAE, 2,277.88 EUR, against 2,420.37 at 0.03 and 2,401.46 at 0.3.** At 0.3 training MAE drops to 1,091.98 EUR.

__Step 57:__ Change only histogram boosting `max_leaf_nodes` on the fixed training folds, with the sweep list `[7, 15, 31]`.

### `histogram boosting: max_leaf_nodes`

The tree repeatedly splits the leaf giving the largest loss reduction until reaching this cap. The default is `31`. More leaves are expected to capture finer structure at greater cost, with a greater chance of fitting noise. The candidate list is `[7, 15, 31]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
hist_max_leaf_nodes_values = [7, 15, 31]
# One row per value will hold its scores, its seconds and its fitted model.
hist_max_leaf_nodes_rows = []
for value in tqdm(hist_max_leaf_nodes_values, desc="HistGradientBoostingRegressor.max_leaf_nodes", unit="value"):
    started = time.perf_counter()
    candidate = clone(hist_reference).set_params(max_leaf_nodes=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    hist_max_leaf_nodes_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
hist_max_leaf_nodes_sweep = pd.DataFrame(hist_max_leaf_nodes_rows)
hist_complete_rows.extend(hist_max_leaf_nodes_rows)
hist_max_leaf_nodes_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
hist_max_leaf_nodes_winner = hist_max_leaf_nodes_sweep.loc[hist_max_leaf_nodes_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": hist_max_leaf_nodes_winner["configuration"], "train": hist_max_leaf_nodes_winner["train"],
    "validation": hist_max_leaf_nodes_winner["validation"], "model": hist_max_leaf_nodes_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**More leaves per tree help: validation MAE is 2,427.40 EUR at 7, 2,327.36 at 15 and 2,277.88 at 31.**

__Step 58:__ Change only histogram boosting `l2_regularization` on the fixed training folds, with the sweep list `[0.0, 1.0, 10.0]`.

### `histogram boosting: l2_regularization`

This penalty enlarges the denominator used to calculate a leaf value, shrinking its correction. The default is `0.0`, with no penalty. A larger value is expected to weaken corrections most in leaves with few rows or low loss curvature. The candidate list is `[0.0, 1.0, 10.0]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
hist_l2_regularization_values = [0.0, 1.0, 10.0]
# One row per value will hold its scores, its seconds and its fitted model.
hist_l2_regularization_rows = []
for value in tqdm(hist_l2_regularization_values, desc="HistGradientBoostingRegressor.l2_regularization", unit="value"):
    started = time.perf_counter()
    candidate = clone(hist_reference).set_params(l2_regularization=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    hist_l2_regularization_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
hist_l2_regularization_sweep = pd.DataFrame(hist_l2_regularization_rows)
hist_complete_rows.extend(hist_l2_regularization_rows)
hist_l2_regularization_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
hist_l2_regularization_winner = hist_l2_regularization_sweep.loc[hist_l2_regularization_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": hist_l2_regularization_winner["configuration"], "train": hist_l2_regularization_winner["train"],
    "validation": hist_l2_regularization_winner["validation"], "model": hist_l2_regularization_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**Validation MAE is 2,277.88 EUR, 2,278.58 and 2,279.11 at 0.0, 1.0 and 10.0.**

__Step 59:__ Change only histogram boosting `max_bins` on the fixed training folds, with the sweep list `[63, 127, 255]`.

### `histogram boosting: max_bins`

Numeric values are grouped before fitting, and trees can split only between bins. The default is `255`, plus a separate bin for missing values. Fewer bins are expected to fit faster but may remove a useful threshold. The candidate list is `[63, 127, 255]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
hist_max_bins_values = [63, 127, 255]
# One row per value will hold its scores, its seconds and its fitted model.
hist_max_bins_rows = []
for value in tqdm(hist_max_bins_values, desc="HistGradientBoostingRegressor.max_bins", unit="value"):
    started = time.perf_counter()
    candidate = clone(hist_reference).set_params(max_bins=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    hist_max_bins_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
hist_max_bins_sweep = pd.DataFrame(hist_max_bins_rows)
hist_complete_rows.extend(hist_max_bins_rows)
hist_max_bins_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
hist_max_bins_winner = hist_max_bins_sweep.loc[hist_max_bins_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": hist_max_bins_winner["configuration"], "train": hist_max_bins_winner["train"],
    "validation": hist_max_bins_winner["validation"], "model": hist_max_bins_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**Validation MAE is 2,276.56 EUR with 63 bins, 2,278.42 with 127 and 2,277.88 with 255.**

__Step 60:__ Change only histogram boosting `loss` on the fixed training folds, with the sweep list `['squared_error', 'absolute_error', 'quantile']`.

### `histogram boosting: loss`

Loss defines the negative gradient fitted by each tree; the default is `squared_error`. Absolute error targets the median, as does quantile loss at the fixed `quantile=0.5` used throughout this conditional sweep. These alternatives are expected to reduce the influence of extreme target values. The candidate list is `['squared_error', 'absolute_error', 'quantile']`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
hist_loss_reference = clone(hist_reference).set_params(quantile=0.5)
hist_loss_values = ['squared_error', 'absolute_error', 'quantile']
# One row per value will hold its scores, its seconds and its fitted model.
hist_loss_rows = []
for value in tqdm(hist_loss_values, desc="HistGradientBoostingRegressor.loss", unit="value"):
    started = time.perf_counter()
    candidate = clone(hist_loss_reference).set_params(loss=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared]
        + [joblib.delayed(clone(candidate).fit)(Z_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(prepared, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    hist_loss_rows.append({"configuration": " ".join(str(candidate).split()), "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(Z_test)), "train": train_scores,
        "validation": validation_scores, "parameters": candidate.get_params(deep=False), "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
hist_loss_sweep = pd.DataFrame(hist_loss_rows)
hist_complete_rows.extend(hist_loss_rows)
hist_loss_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
hist_loss_winner = hist_loss_sweep.loc[hist_loss_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": hist_loss_winner["configuration"], "train": hist_loss_winner["train"],
    "validation": hist_loss_winner["validation"], "model": hist_loss_winner["model"]},
    Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**The absolute error scores 2,239.13 EUR validation MAE against 2,277.88 for the squared error, and the median quantile gives the same 2,239.13.** The two losses target the same conditional median.

__Step 61:__ Plot training and validation MAE for every other swept histogram boosting setting, one panel each.

In [ ]:
# The sweep tables of this family, each with its list of swept values for the axis labels.
hist_settings_tables = {"learning_rate": (hist_learning_rate_sweep, hist_learning_rate_values), "max_leaf_nodes": (hist_max_leaf_nodes_sweep, hist_max_leaf_nodes_values), "l2_regularization": (hist_l2_regularization_sweep, hist_l2_regularization_values), "max_bins": (hist_max_bins_sweep, hist_max_bins_values), "loss": (hist_loss_sweep, hist_loss_values)}
# One panel per setting: training and validation score at every swept value, side by side.
fig, axes = plt.subplots(2, 3, figsize=(12, 7.2), squeeze=False)
for axis, (setting, (table, values)) in zip(axes.ravel(), hist_settings_tables.items()):
    positions = np.arange(len(table))
    axis.plot(positions, table["train_MAE"], marker="o", color=PLOT_ORANGE, label="training")
    axis.plot(positions, table["cv_MAE"], marker="o", color=PLOT_BLUE, label="validation")
    axis.set(title=setting, xticks=positions, xticklabels=[str(value) for value in values], ylabel="MAE")
    axis.tick_params(axis="x", labelrotation=20)
axes.ravel()[0].legend()
# Hide the panels left over in the last row.
for axis in axes.ravel()[len(hist_settings_tables):]:
    axis.set_visible(False)
plt.tight_layout()
plt.show()

**Validation MAE stays between 2,239.13 EUR and 2,427.40 across the five panels, lowest with the absolute error loss and highest with 7 leaves per tree.** Training MAE falls to 1,091.98 EUR at a learning rate of 0.3, the widest gap between the two lines of any panel.

__Step 62:__ Draw the validation MAE of histogram boosting at every stage for four learning rates.

### `histogram boosting: learning rate and iteration count`

Smaller corrections often need more stages, but noise, overfitting and early termination can break that pattern. The first study plots actual fitted stages against validation score at each learning rate, with an explicit larger stage budget. The second enables a stopping criterion; validation determines whether its chosen count helps.

In [ ]:
# Four learning rates, each with its own staged validation curve (mean over the ten folds).
hist_interaction_reference = clone(hist_reference).set_params(early_stopping=False)
hist_curve_rates = [0.03, 0.1, 0.3, 1.0]
hist_rate_curves = {}
# Fit every fold once per rate at the largest stage count, then score the prediction after each stage: one fit gives the whole curve.
for rate in tqdm(hist_curve_rates, desc="HistGradientBoostingRegressor learning rates", unit="rate"):
    fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(hist_interaction_reference).set_params(learning_rate=rate, max_iter=300).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
    hist_rate_curves[rate] = np.mean([[mean_absolute_error(fold["y_score"], on_score) for on_score in fitted.staged_predict(fold["X_score"])] for fold, fitted in zip(prepared, fits)], axis=0)
# Draw the curves; the dot marks the stage with the best validation score of each rate.
fig, ax = plt.subplots(figsize=(7, 5))
for rate, curve in hist_rate_curves.items():
    ax.plot(np.arange(1, len(curve) + 1), curve, linewidth=2, label=f"learning rate {rate}")
    ax.scatter(np.argmin(curve) + 1, curve.min(), zorder=3)
ax.set(xlabel="max_iter", ylabel="validation MAE", title="HistGradientBoostingRegressor: validation by stage at four learning rates")
ax.legend()
plt.tight_layout()
plt.show()
# Table of the same curves: the best stage, the stage where each curve first comes close to its best, and that best score.
pd.DataFrame({"best stage": [int(np.argmin(curve)) + 1 for curve in hist_rate_curves.values()],
    "stages to come within 1 percent of the best": [int(np.argmax(curve <= 1.01 * curve.min())) + 1 for curve in hist_rate_curves.values()],
    "best validation MAE": [curve.min() for curve in hist_rate_curves.values()]}, index=pd.Index(hist_curve_rates, name="learning_rate")).round(4)

**A rate of 0.03 reaches validation MAE 2,258.89 EUR at stage 299, a rate of 0.1 reaches 2,276.68 at stage 83, a rate of 0.3 reaches 2,358.84 at stage 27 and a rate of 1.0 reaches 2,828.41 at stage 13.** Within 1 percent of its best, the rate 0.03 needs 181 stages, the rate 0.1 needs 52, the rate 0.3 needs 14 and the rate 1.0 needs 4.

__Step 63:__ Stop each histogram boosting fit with its convergence criterion and count the stages it used.

In [ ]:
# With a convergence criterion each fit stops by itself, so the learning rate decides how many stages are fitted.
hist_stop_rows = []
for rate in tqdm(hist_curve_rates, desc="HistGradientBoostingRegressor early stopping", unit="rate"):
    started = time.perf_counter()
    fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(hist_reference).set_params(learning_rate=rate, max_iter=1000, early_stopping=True, n_iter_no_change=10, validation_fraction=0.1).fit)(fold["X_fit"], fold["y_fit"]) for fold in prepared)
    hist_stop_rows.append({"learning_rate": rate, "stages fitted": np.mean([fitted.n_iter_ for fitted in fits]),
        "validation MAE": np.mean([mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(prepared, fits)]),
        "seconds": time.perf_counter() - started})
pd.DataFrame(hist_stop_rows).set_index("learning_rate").round(4)

**With `early_stopping` on, the fit stops after 195.0 iterations on average over the folds at a rate of 0.03, 82.9 at 0.1, 33.8 at 0.3 and 19.1 at 1.0.** The validation MAE is 2,290.14 EUR, 2,298.07, 2,377.21 and 2,974.42, so a smaller rate stops later and has a lower MAE here.

__Step 64:__ Select the best tested histogram boosting configuration from the existing sweep rows by validation score.

In [ ]:
hist_complete = pd.DataFrame(hist_complete_rows)
hist_best = hist_complete.loc[hist_complete["cv_MAE"].idxmin()]
hist_best_parameters = hist_best["parameters"]
hist_best_configuration = clone(hist_reference).set_params(**hist_best_parameters)
print("best tested histogram boosting configuration:", hist_best["configuration"])

**The best complete histogram boosting configuration measured uses the absolute error loss, with validation MAE 2,239.13 EUR.**

# <font color='#E8800A'>Selection of the boosting model</font> <a class="anchor" id="sel1"></a>
[Back to TOC](#toc)

The three boosting families are selected together by one search. Nested cross-validation comes first: it scores the whole search procedure on outer-fold rows that the search never saw, which checks that the grid is viable. The search on all the training rows then picks the boosting winner from that grid, and that model joins the combiners below.

The grid crosses all swept predictive dimensions of the three boosting families and the tree settings listed in the code. Other leaf-growth and regularisation controls retain documented defaults. Execution controls, seeds and warm starts stay fixed; disabling histogram early stopping makes stage counts comparable across folds. Domain constraints and native categorical handling require a feature-specific course hypothesis.

The search draws 60 combinations uniformly without replacement from the full enumerated space, using seed 33 before scoring. The same 60 feed the nested estimate and the all-training search. Bergstra and Bengio (2012) show that 60 random draws reach the best 5 percent with about 95 percent probability ([Random search for hyper-parameter optimization](https://jmlr.org/papers/v13/bergstra12a.html)). With list-valued parameters, scikit-learn's [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) uses `ParameterSampler` to sample uniformly without replacement over `ParameterGrid` combinations: each has equal probability. `CachedSearchCV` enables preparation caching to avoid repeated preparation.

The search space crosses, within each family, every value swept in that family's section: the value lists are read from the sweep cells above, not retyped. The search also draws the tree settings that a single decision tree takes, with the values listed in the code.

__Step 65:__ Declare the conditional grids of the boosting families, which fix the search space before any search score exists.

In [ ]:
# The search space of the first selection: for each boosting family one conditional Cartesian product of the value lists swept above, read from the sweep cells and declared before any search score exists. Each block names its own preparation step.
# The value lists of the tree settings. The sweeps above keep them at their reference values, so they enter the search here and nowhere else. The pruning strength stays at its reference: its price-scale values leave the stage trees of the absolute error and quantile losses with one leaf.
tree_values = {
    "gb": {"min_samples_leaf": [1, 5, 20, 50], "min_samples_split": [2, 10, 50], "max_features": [None, "sqrt", "log2"]},
    "hist": {"max_depth": [3, 5, 8, 12, None], "min_samples_leaf": [1, 5, 20, 50]}}
boost_grid = [
    {"preprocessor": [clone(preparation)], "estimator": [clone(ada_reference)],
     "estimator__estimator": ada_estimator_values,
     "estimator__n_estimators": ada_n_estimators_values,
     "estimator__learning_rate": ada_learning_rate_values,
     "estimator__loss": ada_loss_values},
    # Gradient boosting: every combination of the settings swept in its section; a setting that only some losses use sits in its own block.
    {"preprocessor": [clone(preparation)], "estimator": [clone(gb_reference)],
     "estimator__n_estimators": gb_n_estimators_values,
     "estimator__learning_rate": gb_learning_rate_values,
     "estimator__max_depth": gb_max_depth_values,
     "estimator__min_samples_leaf": tree_values["gb"]["min_samples_leaf"],
     "estimator__min_samples_split": tree_values["gb"]["min_samples_split"],
     "estimator__subsample": gb_subsample_values,
     "estimator__max_features": tree_values["gb"]["max_features"],
     "estimator__loss": [loss for loss in gb_loss_values if loss not in ("huber", "quantile")]},
    {"preprocessor": [clone(preparation)], "estimator": [clone(gb_reference)],
     "estimator__n_estimators": gb_n_estimators_values,
     "estimator__learning_rate": gb_learning_rate_values,
     "estimator__max_depth": gb_max_depth_values,
     "estimator__min_samples_leaf": tree_values["gb"]["min_samples_leaf"],
     "estimator__min_samples_split": tree_values["gb"]["min_samples_split"],
     "estimator__subsample": gb_subsample_values,
     "estimator__max_features": tree_values["gb"]["max_features"],
     "estimator__loss": [loss for loss in gb_loss_values if loss in ("huber", "quantile")],
     "estimator__alpha": gb_alpha_values},
    # Histogram boosting: the same crossing for its settings.
    {"preprocessor": [clone(preparation)], "estimator": [clone(hist_reference)],
     "estimator__max_iter": hist_max_iter_values,
     "estimator__learning_rate": hist_learning_rate_values,
     "estimator__max_leaf_nodes": hist_max_leaf_nodes_values,
     "estimator__max_depth": tree_values["hist"]["max_depth"],
     "estimator__min_samples_leaf": tree_values["hist"]["min_samples_leaf"],
     "estimator__l2_regularization": hist_l2_regularization_values,
     "estimator__max_bins": hist_max_bins_values,
     "estimator__loss": [loss for loss in hist_loss_values if loss != "quantile"]},
    {"preprocessor": [clone(preparation)], "estimator": [clone(hist_reference)],
     "estimator__max_iter": hist_max_iter_values,
     "estimator__learning_rate": hist_learning_rate_values,
     "estimator__max_leaf_nodes": hist_max_leaf_nodes_values,
     "estimator__max_depth": tree_values["hist"]["max_depth"],
     "estimator__min_samples_leaf": tree_values["hist"]["min_samples_leaf"],
     "estimator__l2_regularization": hist_l2_regularization_values,
     "estimator__max_bins": hist_max_bins_values,
     "estimator__loss": [loss for loss in hist_loss_values if loss == "quantile"],
     "estimator__quantile": [0.5]}
]

__Step 66:__ Enumerate every valid combination, and draw 60 of them without replacement before any score is seen.

In [ ]:
# The conditional grids hold every valid combination; the search draws 60 of them, uniformly and without replacement, before any score exists.
print("combinations by group:", [len(ParameterGrid(group)) for group in boost_grid])
print("combinations in the space:", len(ParameterGrid(boost_grid)))

**The conditional grids of the boosting families hold 126900 valid combinations, and the search evaluates 60 of them.**

<div class="alert alert-block alert-info">
<code>CachedSearchCV(search)</code>

A course class in `preprocessing.py`, beside this notebook.

**Definition:**
Wraps a scikit-learn search over a `PreparedEstimator`, fitting each distinct preprocessing setting once per training-row set across candidates. Unlike the Week 5 class, which implemented its own grid search, this wrapper accepts any scikit-learn search and retains all its results.

**Main Parameters:**
- `search`: an unfitted scikit-learn search, such as `GridSearchCV`, `RandomizedSearchCV` or `HalvingGridSearchCV`, whose estimator is a `PreparedEstimator`.

**Main Attributes (after fit):**
- `cv_results_`, `best_params_`, `best_score_`, `best_index_`, `best_estimator_`: those of the search.
- `search_`: the fitted search itself.

**Main Methods:**
- `fit(X, y)`: run the search with the cache and keep its results.
- `predict(X)`, `predict_proba(X)`, `score(X, y)`: those of the best estimator, refitted on all the rows.

**Differences from scikit-learn:**
Alone, [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) repeats preprocessing per candidate and fold. This wrapper adds a temporary disk cache keyed by settings and rows. The grid and random searches with integer `random_state` prefit distinct settings once per fold; subsequent search fits read the cache. For adaptive searches such as `HalvingGridSearchCV` lock concurrent requests around one fit. Candidates, scores and winner remain unchanged. Estimators without `memory` run as they would unwrapped.
</div>

__Step 67:__ Wrap the grid and the preprocessing in one search, with three inner folds.

In [ ]:
inner = KFold(n_splits=3, shuffle=True, random_state=random_state)
boost_template = CachedSearchCV(RandomizedSearchCV(PreparedEstimator(FunctionTransformer(), clone(ada_reference)), boost_grid, n_iter=60, random_state=random_state, cv=inner, scoring="neg_mean_absolute_error", n_jobs=4))

__Step 68:__ Count the candidates and the model fits the search can cost.

In [ ]:
boost_count = boost_template.search.n_iter
print("inner-fold candidate fits per search:", boost_count * inner.get_n_splits())
print("outer splits:", folds.get_n_splits(), "inner splits:", inner.get_n_splits())
print("candidate fits plus winner refits:", (folds.get_n_splits() + 1) * (boost_count * inner.get_n_splits() + 1))

__Step 69:__ Estimate the whole search with the ten outer folds of the training partition, returning scores and the settings each outer search chose.

In [ ]:
boost_nested_started = time.perf_counter()
# Each of the ten outer folds runs the whole search on its own training part, one fold per worker; the search inside uses four workers.
outer_runs = joblib.Parallel(n_jobs=10, return_as="generator")(
    joblib.delayed(cross_validate)(boost_template, X_train, y_train, cv=[split],
        scoring="neg_mean_absolute_error", return_train_score=True, n_jobs=1, return_estimator=True, error_score="raise")
    for split in folds.split(X_train, y_train))
# Collect the outer results as the workers finish; the progress bar fills in bursts.
boost_fold_results = list(tqdm(outer_runs, total=folds.get_n_splits(),
                               desc="Nested CV, boosting: outer folds", unit="fold", leave=True, position=0))
# The outer validation scores estimate the whole procedure, selection step included; the training scores sit beside them.
boost_nested_scores = {key: np.concatenate([result[key] for result in boost_fold_results]) for key in ("train_score", "test_score")}
boost_nested_validation = -boost_nested_scores["test_score"]
boost_nested_train = -boost_nested_scores["train_score"]
# The settings each outer search chose are kept as plain settings; the fitted searches are dropped.
boost_fold_settings = [result["estimator"][0].best_params_ for result in boost_fold_results]
boost_nested_seconds = time.perf_counter() - boost_nested_started
# Report the estimate, its spread over the folds and the time.
print(f"nested validation MAE: {boost_nested_validation.mean():.4f}")
print(f"fold SD: {boost_nested_validation.std(ddof=1):.4f}")
print(f"nested wall time: {boost_nested_seconds:.2f} seconds")

**The complete search scores a nested validation MAE of 2,237.74 EUR with a fold SD of 117.01.** The space holds 126900 valid combinations, 60 of them drawn for the search.

# <font color='#E8800A'>The boosting winner</font> <a class="anchor" id="boost_winner"></a>
[Back to TOC](#toc)

With the search space fixed and its nested estimate available, the same candidates are fitted to all the training rows. Its best estimator is the boosting winner: it supplies the predictions, the model artifact and one row of the course board, even when it differs from a default or from a sweep winner.

__Step 70:__ Run the declared grid of the boosting families on all the training rows and keep its winner.

In [ ]:
boost_started = time.perf_counter()
boost_search = clone(boost_template).fit(X_train, y_train)
boosting_winner = boost_search.best_estimator_
boost_seconds = time.perf_counter() - boost_started
print("chosen parameters:", boost_search.best_params_)
print("winner own inner-CV score:", boost_search.best_score_)
print("combinations evaluated:", len(boost_search.cv_results_["params"]))
print(f"search wall time: {boost_seconds:.2f} seconds")

__Step 71:__ Score the boosting winner on the ten outer folds, and log it beside its test score.

In [ ]:
# Score the winner on the same folds: its settings are fitted on each prepared training part.
boost_folds = prepared
boost_train, boost_validation = [], []
for fold in tqdm(boost_folds, desc='prepared', leave=True, position=0):
    fitted = clone(boosting_winner.estimator).fit(fold["X_fit"], fold["y_fit"])
    boost_train.append(mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])))
    boost_validation.append(mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])))

In [ ]:
# The boosting winner as a plain constructor: the estimator of the winning block with the settings the search chose.
boost_settings = boost_search.best_params_
boosting_member = clone(boost_settings["estimator"]).set_params(**{key.split("estimator__", 1)[1]: value for key, value in boost_settings.items() if key.startswith("estimator__")})
# Name the winner by the settings that differ from the defaults.
boost_label = " ".join(str(boosting_member).split())
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": boost_label, "train": boost_train, "validation": boost_validation, "model": boosting_winner},
    X_test, y_test, "winner's own score", metric=mean_absolute_error, label="MAE")

__Step 72:__ Log the boosting winner with its nested estimate as the validation score.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": f"boosting search over {len(boost_search.cv_results_['params'])} combinations", "train": boost_nested_train, "validation": boost_nested_validation, "model": boosting_winner},
    X_test, y_test, "nested estimate", metric=mean_absolute_error, label="MAE")

__Step 73:__ Read the test score of the boosting winner from the log.

In [ ]:
boost_test = model_log[-1]["test MAE"]

**The search chose gradient boosting with the Huber loss, 130 stages, a depth of 5, a subsample of 0.8, square-root feature sampling, 5 rows per leaf and a Huber quantile of 0.95, with held-out MAE 2,260.15 EUR and an inner-CV score of 2,243.34.** Its own score over the ten outer folds is 2,237.74, against a nested estimate of 2,237.74 for the whole search; the search chose the winner on the rows behind the first number, so the nested estimate is the one that describes the procedure.

__Step 74:__ Read every logged attempt before saving the result.

In [ ]:
print(pd.DataFrame(model_log).to_string(index=False))

# <font color='#E8800A'>The boosting result</font> <a class="anchor" id="board"></a>
[Back to TOC](#toc)

The nested estimate includes model selection. The fitted winner and each outer-training selection are saved for the combining notebook.

<div class="alert alert-block alert-info">
<code>save_boosting_selection(path, payload, model)</code>

A helper in `preprocessing.py`, beside this notebook.

**Definition:** Saves the selected parameter dictionaries, fold scores and fitted winner.

**Main Parameters:** `path` names the selection file; `payload` contains task, context, folds, settings and scores; `model` is the fitted winner.

**Returns:** None.

**Differences from scikit-learn:** A search keeps settings and scores in memory. This course helper writes them with task and fold fingerprints, using [joblib](https://joblib.readthedocs.io/en/latest/user_guide/persistence.html), so another notebook can validate and reuse them.
</div>

__Step 75:__ Save the boosting selection and fitted winner with their data and fold fingerprints.

In [ ]:
# Fingerprint the data, recipe, split and exact outer rows.
selection_context = result_context(
    "../../data/interim/cars4you.csv", recipe_log, y_train.index.tolist(), y_test.index.tolist(),
    outer_folds=outer_folds(folds, X_train, y_train))
fold_fingerprint = joblib.hash(outer_folds(folds, X_train, y_train))
boosting_dir = Path(results_dir) / "boosting"
boosting_dir.mkdir(parents=True, exist_ok=True)
# Compare fixed settings with the same family's default on the same ten folds.
boost_family = type(boosting_winner.estimator).__name__
boost_reference = next(row for row in model_log
    if row.get("validation MAE is") == "untuned" and row["model"].startswith(boost_family + "("))
boost_baseline_cv = float(boost_reference["validation MAE"])
boost_baseline_test = float(boost_reference["test MAE"])
boost_tuning_comparison = dict(
    protocol="fixed_settings_on_course_folds", family=boost_family,
    candidate_model=boost_label, reference_model=boost_reference["model"],
    reference_kind="library_default", candidate_cv=float(np.mean(boost_validation)),
    reference_cv=boost_baseline_cv, candidate_test=float(boost_test), reference_test=boost_baseline_test)
# Keep only the selected settings and the scores the combining part needs.
selection_payload = {
    "task": "regression", "context": selection_context, "fold_fingerprint": fold_fingerprint,
    "boost_settings": boost_settings, "boost_fold_settings": boost_fold_settings,
    "boost_nested_scores": dict(train_score=boost_nested_train, test_score=boost_nested_validation),
    "boost_nested_validation": boost_nested_validation, "boost_nested_train": boost_nested_train,
    "boost_train": boost_train, "boost_validation": boost_validation,
    "boost_label": boost_label, "boost_test": float(boost_test),
    "baseline_cv": boost_baseline_cv, "baseline_test": boost_baseline_test,
    "tuning_comparison": boost_tuning_comparison,
}
save_boosting_selection(boosting_dir / "week_11_regression_selection.joblib", selection_payload, boosting_winner)
print("Saved boosting selection:", boosting_dir / "week_11_regression_selection.joblib")

__Step 76:__ Save and read the boosting result.

In [ ]:
board = Scoreboard("cars4you price", "MAE (EUR)", context=result_context(
    "../../data/interim/cars4you.csv", recipe_log, y_train.index.tolist(), y_test.index.tolist(),
    outer_folds=outer_folds(folds, X_train, y_train)),
    floor=mean_absolute_error(y_test, DummyRegressor(strategy="mean").fit(X_train, y_train).predict(X_test)))
for week, earlier_model, earlier_cv, earlier_test in results_before(11, "regression", results_dir, metric=board.metric, context=board.context):
    board.record(week, earlier_model, earlier_cv, earlier_test)
board.record(11, boost_label, round(float(np.mean(boost_nested_validation)), 4), round(float(boost_test), 4))
board.save(boosting_dir, 11, "regression", baseline_cv=round(boost_baseline_cv, 4), baseline_test=round(boost_baseline_test, 4))
# Keep the fixed-setting comparison separate from the nested procedure estimate.
boost_record_path = boosting_dir / "week_11.json"
boost_record = json.loads(boost_record_path.read_text(encoding="utf-8"))
boost_record["regression"]["tuning_comparison"] = boost_tuning_comparison
boost_record_path.write_text(json.dumps(boost_record, indent=1) + "\n", encoding="utf-8")
print(board.table())

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. AdaBoost starts from its library defaults; each isolated sweep changes only its named setting.
2. AdaBoost reweights rows; gradient boosting fits trees to loss gradients.
3. Smaller corrections often need more stages, but validation determines the useful stopping point.
4. A nested estimate scores the search procedure, including its selection step.

# <font color='#E8800A'>Exit ticket</font> <a class="anchor" id="exit"></a>
[Back to TOC](#toc)

At learning rate 0.1, AdaBoost reaches its best plotted validation MAE of 3,833.38 EUR after 78 stages; rate 0.5 reaches 3,830.79 EUR after 11 stages. Why does the smaller rate need more stages to reach a similar error?

**Answer.** A smaller learning rate reduces each stage's influence, so more stages may be needed to reach a similar fit. Validation still determines whether those extra stages help.

# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

Original papers support the method explanations; library documentation defines the estimator contracts.

- [scikit-learn ensemble methods](https://scikit-learn.org/stable/modules/ensemble.html)
- [AdaBoostRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostRegressor.html)
- [GradientBoostingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html)
- [HistGradientBoostingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html)
- [Freund, Y. and Schapire, R. E. (1997). A Decision-Theoretic Generalization of On-Line Learning and an Application to Boosting. Journal of Computer and System Sciences, 55(1), 119-139.](https://doi.org/10.1006/jcss.1997.1504)
- [Friedman, J. H., Hastie, T. and Tibshirani, R. (2000). Additive Logistic Regression: A Statistical View of Boosting. Annals of Statistics, 28(2), 337-407.](https://doi.org/10.1214/aos/1016218223)
- [Friedman, J. H. (2001). Greedy Function Approximation: A Gradient Boosting Machine. Annals of Statistics, 29(5), 1189-1232.](https://doi.org/10.1214/aos/1013203451)
- [Drucker, H. (1997). Improving Regressors using Boosting Techniques. Proceedings of ICML, 107-115.](https://mlanthology.org/icml/1997/drucker1997icml-improving/)
- [Bergstra, J. and Bengio, Y. (2012). Random Search for Hyper-Parameter Optimization. Journal of Machine Learning Research, 13(10), 281-305.](https://jmlr.org/papers/v13/bergstra12a.html)
- [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)
- [Nested versus non-nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)
- [joblib disk-based caching](https://joblib.readthedocs.io/en/latest/user_guide/memory.html)
- [joblib serialization and loading](https://joblib.readthedocs.io/en/latest/user_guide/persistence.html)